# 01 · Reference data from the LI.FI API and CoinGecko

Fetches, once, the reference data the profile could not settle from the two CSVs and snapshots it under `data/reference/`: LI.FI chains, tokens, tools, `/v1/status` for a random sample of the transactions in the data, and CoinGecko hourly prices for every (chain, token) pair that appears as `sendingAssetId` in `transfers_started.csv`. §2.1 adds LI.FI's chain ids across its published sources: the API, `LiFiData.sol` in the `lifinance/contracts` repository, and the `@lifi/types` and `@lifi/data-types` npm packages, fetched without a key.

Rules applied: every fetch cell reads the snapshot if it exists and only calls the network when it does not. The raw files are not in this repository, so the committed outputs are the record of the run. `data/reference/snapshot_manifest.json` records source URL, fetch time (UTC) and row count per file at fetch time; `data/reference/README.md` is written from it. The CoinGecko key, if any, is read from the environment variable `COINGECKO_API_KEY`, else from a `COINGECKO_API_KEY=…` line in the gitignored `.env` at the repository root; it is sent as a header and never printed or written. Every number below that depends on reference data says so.

Sections: 1 setup and the two CSVs · 2 chains (2.1 chain ids across LI.FI's published sources) · 3 tokens · 4 tools · 5 status sample · 6 CoinGecko prices · 7 destination ids resolved · 8 tokens resolved (unknown tokens, decimals, symbols, USDC variants) · 9 bridge strings against tools · 10 the status sample: completion, amounts, gas, fees, tools · 11 manifest.

## 1. Setup and the two CSVs

Loading as in `00_explore.ipynb` §1: `bridge_data` parsed with `json.loads` so `minAmount` stays an exact integer; `hour` is `block_time` floored to the hour. The column `amount_usd_provided` is the profile's first USD column (`00_explore.ipynb` §11: `minAmount / 10^decimals × price_usd` from `prices.csv` only, no fallback) and is used here only for USD shares; the headline column with the CoinGecko fill is built in `02_activity_and_prices.ipynb` §8.

The HTTP helper backs off on HTTP 429 and 5xx (honouring `Retry-After`), and `snapshot_csv` implements the read-if-exists rule.

In [1]:
import json
import os
import time
import datetime as dt
from decimal import Decimal
from pathlib import Path

import pandas as pd
import requests

from lifi import paths

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 60)

ROOT = paths.REPO_ROOT                 # the repository root, where the gitignored .env lives
DATA = paths.data_dir()                # LIFI_DATA_DIR, else data/ in the repository; only the two CSVs are read from it
DATA_LABEL = "LIFI_DATA_DIR" if os.environ.get("LIFI_DATA_DIR") else "data under the repository root"   # printed instead of a path
REF = paths.reference_dir()            # data/reference in the repository, whatever LIFI_DATA_DIR says: snapshots are read from it and, when missing, fetched into it
REF.mkdir(exist_ok=True)
TRANSFERS, PRICES = DATA / paths.TRANSFERS_FILE, DATA / paths.PRICES_FILE
for f in (TRANSFERS, PRICES):
    if not f.exists():
        raise FileNotFoundError(f"{f.name} is missing from the data directory ({DATA_LABEL}): drop the provided CSV there (see analysis/README.md)")
MANIFEST = REF / "snapshot_manifest.json"

ZERO_ADDRESS = "0x0000000000000000000000000000000000000000"
SENTINEL_RECEIVER = "0x11f111f111f111f111f111f111f111f111f111f1"   # the receiver placeholder the profile found on the large destination ids
EXPORT_START = pd.Timestamp("2026-10-13 00:00:00", tz="UTC")       # inclusive, data/README_data_final.md
EXPORT_END = pd.Timestamp("2026-10-20 00:00:00", tz="UTC")         # exclusive

LIFI = "https://li.quest/v1"
COINGECKO = "https://api.coingecko.com/api/v3"
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "lifi-takehome-analysis/0.1 (analysis notebook)"


def read_env_key(name):
    "The key from the environment, else from a KEY=VALUE line in the gitignored .env at the repository root; None if neither has it. The value is never printed or written."
    value = os.environ.get(name)
    if value:
        return value
    env_file = ROOT / ".env"
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            line = line.strip()
            if line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            if k.strip() == name:
                return v.strip().strip('"').strip("'") or None
    return None


COINGECKO_KEY = read_env_key("COINGECKO_API_KEY")   # read only; never printed, never written
print("data dir:", DATA_LABEL, "| CoinGecko key present:", bool(COINGECKO_KEY))


def now_utc():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


THROTTLE = {"count": 0}   # how many HTTP 429 responses the helper has seen; the CoinGecko loop slows down on it


def get_json(url, params=None, headers=None, pause=0.0, max_tries=8, timeout=60):
    "GET with backoff on HTTP 429, 5xx and network errors (timeouts, dropped connections). Returns (status_code, parsed JSON or text); status 0 when every try failed on the network. Sleeps `pause` seconds after a successful call."
    wait = 5.0
    r = None
    for attempt in range(max_tries):
        try:
            r = SESSION.get(url, params=params, headers=headers, timeout=timeout)
        except requests.RequestException as e:
            print(f"  network error ({url.split('/v1/')[-1].split('/v3/')[-1][:50]}): {type(e).__name__}; sleeping {wait:.0f}s, attempt {attempt + 1}/{max_tries}")
            time.sleep(wait)
            wait = min(wait * 2, 120)
            if attempt == max_tries - 1:
                return 0, f"network error after {max_tries} tries: {type(e).__name__}"
            continue
        if r.status_code == 429 or r.status_code >= 500:
            THROTTLE["count"] += r.status_code == 429
            retry_after = r.headers.get("Retry-After")
            sleep_s = float(retry_after) if retry_after and retry_after.isdigit() else wait
            print(f"  HTTP {r.status_code} ({url.split('/v1/')[-1].split('/v3/')[-1][:50]}): sleeping {sleep_s:.0f}s, attempt {attempt + 1}/{max_tries}")
            time.sleep(sleep_s + 1)
            wait = min(wait * 2, 120)
            continue
        if pause:
            time.sleep(pause)
        try:
            return r.status_code, r.json()
        except ValueError:
            return r.status_code, r.text
    return r.status_code, r.text


def manifest_read():
    return json.loads(MANIFEST.read_text()) if MANIFEST.exists() else {}


def manifest_write(name, source_url, rows, section, fetched_at, note=None):
    m = manifest_read()
    m[name] = {"source_url": source_url, "fetched_at_utc": fetched_at, "rows": int(rows), "notebook_section": section}
    if note:
        m[name]["note"] = note
    MANIFEST.write_text(json.dumps(m, indent=1, sort_keys=True) + "\n")


def snapshot_csv(name, fetch, section, source_url, note=None, read_kwargs=None):
    "Read data/reference/<name> if it exists; otherwise call fetch() -> DataFrame, write it and record the manifest entry. `note` may be a callable evaluated after the fetch."
    path = REF / name
    if path.exists():
        df = pd.read_csv(path, **(read_kwargs or {}))
        print(f"{name}: read snapshot, {len(df)} rows, fetched {manifest_read().get(name, {}).get('fetched_at_utc', '?')} (no network call)")
        return df
    print(f"{name}: no snapshot, fetching from {source_url}")
    t0 = now_utc()
    df = fetch()
    df.to_csv(path, index=False)
    manifest_write(name, source_url, len(df), section, t0, note() if callable(note) else note)
    print(f"{name}: fetched {len(df)} rows starting {t0}, written to {REF.relative_to(ROOT)}/{name} in the repository")
    return pd.read_csv(path, **(read_kwargs or {}))


def append_csv(path, df):
    df.to_csv(path, mode="a", index=False, header=not path.exists())

data dir: data under the repository root | CoinGecko key present: False


In [2]:
raw = pd.read_csv(TRANSFERS)
prices = pd.read_csv(PRICES)
bridge = pd.DataFrame([json.loads(s) for s in raw["bridge_data"]])
tf = pd.concat([raw.drop(columns="bridge_data"), bridge], axis=1)
tf["block_time"] = pd.to_datetime(tf["block_time"], utc=True)
tf["hour"] = tf["block_time"].dt.floor("h")
prices["hour"] = pd.to_datetime(prices["hour"], utc=True)


def token_amount(min_amount, decimals):
    if pd.isna(decimals):
        return float("nan")
    return float(Decimal(int(min_amount)) / Decimal(10) ** int(decimals))


# The profile's first USD column (00_explore.ipynb §11): provided feed only, no fallback.
joined = tf.merge(prices[["chain", "token_address", "hour", "decimals", "price_usd"]],
                  left_on=["chain", "sendingAssetId", "hour"], right_on=["chain", "token_address", "hour"], how="left")
tf["amount_usd_provided"] = pd.Series([token_amount(a, d) for a, d in zip(joined["minAmount"], joined["decimals"])], index=joined.index) * joined["price_usd"]
TOTAL_USD_PROVIDED = tf["amount_usd_provided"].sum()
print("transfers:", tf.shape, " prices:", prices.shape)
print("rows priced by the provided feed:", int(tf["amount_usd_provided"].notna().sum()), " unpriced:", int(tf["amount_usd_provided"].isna().sum()),
      f" total USD (provided only): {TOTAL_USD_PROVIDED:,.2f}")

transfers: (124858, 19)  prices: (40754, 6)
rows priced by the provided feed: 124477  unpriced: 381  total USD (provided only): 307,935,992.59


## 2. Chains

`GET /v1/chains`. Without a `chainTypes` parameter the endpoint returns EVM chains only (65 at fetch time), so the fetch passes `EVM,SVM,UTXO,MVM,TVM`. The parameter also accepts a sixth value, `STL` (Stellar), which LI.FI's API reference documents. This fetch did not pass it, so `lifi_api_chains.csv` lacks Stellar. The snapshot stays as fetched, and §2.1 re-fetches with all six values. A value outside the six is rejected with HTTP 400 "must be equal to one of the allowed values" (§2.1 keeps one such response). Snapshot `lifi_api_chains.csv`: one row per chain with id, key, name, chain type, native token symbol and decimals, and the LI.FI Diamond address.

In [3]:
CHAIN_TYPES = "EVM,SVM,UTXO,MVM,TVM"
CHAINS_URL = f"{LIFI}/chains?chainTypes={CHAIN_TYPES}"


def fetch_chains():
    status, body = get_json(f"{LIFI}/chains", params={"chainTypes": CHAIN_TYPES})
    assert status == 200, (status, body)
    rows = []
    for c in body["chains"]:
        nt = c.get("nativeToken") or {}
        rows.append({"id": c["id"], "key": c["key"], "name": c["name"], "chainType": c["chainType"], "coin": c.get("coin"), "mainnet": c.get("mainnet"),
                     "native_symbol": nt.get("symbol"), "native_name": nt.get("name"), "native_decimals": nt.get("decimals"), "native_address": nt.get("address"),
                     "diamondAddress": c.get("diamondAddress")})
    return pd.DataFrame(rows).sort_values("id").reset_index(drop=True)


chains = snapshot_csv("lifi_api_chains.csv", fetch_chains, "01 §2", CHAINS_URL,
                      note="chainTypes names every value the endpoint accepts; without it only EVM chains are returned",
                      read_kwargs={"dtype": {"key": str, "native_address": str, "diamondAddress": str, "native_symbol": str, "coin": str}})
print("chains by type (reference data):", chains["chainType"].value_counts().to_dict())
chains[chains["chainType"] != "EVM"]

lifi_api_chains.csv: read snapshot, 70 rows, fetched 2026-10-28T14:48:32+00:00 (no network call)
chains by type (reference data): {'EVM': 65, 'UTXO': 2, 'TVM': 1, 'SVM': 1, 'MVM': 1}


,id,key,name,chainType,coin,mainnet,native_symbol,native_name,native_decimals,native_address,diamondAddress
65,728126428,trn,Tron,TVM,TRX,True,TRX,TRX,6,T9yD14Nj9j7xAB4dbGeiX9h8unkKHxuWwb,TU3ymitEKCWQFtASkEeHaPb8NfZcJtCHLt
66,20000000000001,btc,Bitcoin,UTXO,BTC,True,BTC,Bitcoin,8,bitcoin,NaN
67,20000000000005,zec,Zcash,UTXO,ZEC,True,ZEC,Zcash,8,zcash,NaN
68,1151111081099710,sol,Solana,SVM,SOL,True,SOL,SOL,9,11111111111111111111111111111111,NaN
69,9270000000000000,sui,Sui,MVM,SUI,True,SUI,SUI,9,0x000000000000000000000000000000000000000000000000000000...,NaN


In [4]:
# The six source chains: the profile assumed these ids (ethereum=1, optimism=10, bnb=56, polygon=137, base=8453, arbitrum=42161); the API names them.
SOURCE_CHAIN_ID = {"ethereum": 1, "arbitrum": 42161, "base": 8453, "optimism": 10, "polygon": 137, "bnb": 56}
src = pd.DataFrame({"chain": list(SOURCE_CHAIN_ID), "id": list(SOURCE_CHAIN_ID.values())}).merge(chains, on="id", how="left")
src["diamond_matches_data"] = src["diamondAddress"].str.lower() == tf["contract_address"].iloc[0]
src[["chain", "id", "key", "name", "chainType", "native_symbol", "native_decimals", "diamondAddress", "diamond_matches_data"]]

,chain,id,key,name,chainType,native_symbol,native_decimals,diamondAddress,diamond_matches_data
0,ethereum,1,eth,Ethereum,EVM,ETH,18,0x1231DEB6f5749EF6cE6943a275A1D3E7486F4EaE,True
1,arbitrum,42161,arb,Arbitrum,EVM,ETH,18,0x1231DEB6f5749EF6cE6943a275A1D3E7486F4EaE,True
2,base,8453,bas,Base,EVM,ETH,18,0x1231DEB6f5749EF6cE6943a275A1D3E7486F4EaE,True
3,optimism,10,opt,OP Mainnet,EVM,ETH,18,0x1231DEB6f5749EF6cE6943a275A1D3E7486F4EaE,True
4,polygon,137,pol,Polygon,EVM,POL,18,0x1231DEB6f5749EF6cE6943a275A1D3E7486F4EaE,True
5,bnb,56,bsc,BSC,EVM,BNB,18,0x1231DEB6f5749EF6cE6943a275A1D3E7486F4EaE,True


### 2.1 Chain ids across LI.FI's published sources

LI.FI uses chain ids in two namespaces. The **API namespace** is what `/v1/chains`, the SDK's `@lifi/types` `ChainId` enum and `@lifi/data-types` `supportedChains` carry. The **contract namespace** is the `destinationChainId` the LI.FI Diamond writes into `BridgeData`, the value in `transfers_started.csv`. For an EVM chain both are the chain's EIP-155 id. For a non-EVM chain LI.FI assigns an id ("IDs are made up by the LI.FI team", `LiFiData.sol`), and the two namespaces agree except for Tron. This section fetches, once and without an API key:

- `GET /v1/chains?chainTypes=EVM,SVM,UTXO,MVM,TVM,STL`, the six values LI.FI's API reference (`https://docs.li.fi/openapi.yaml`) documents for `chainTypes`. §2 passed only the first five, which left out `STL` (Stellar). Snapshot `lifi_api_chains_with_stl_2026-10-28.csv` has the same columns as `lifi_api_chains.csv`, which stays as fetched. One extra call passes an invalid value, to record the validator's message.
- `src/Helpers/LiFiData.sol` from `lifinance/contracts` at commit `f5686943150bf16cb0d1de649caef3cfa4dc959c`, with the GitHub commits API showing whether that commit is still the last one touching the file on `main`. The file defines `NON_EVM_ADDRESS` and the `LIFI_CHAIN_ID_*` constants.
- From npm, `@lifi/types` 18.14.0 (`src/chains/base.ts` for `ChainId` and `ChainKey`, `src/chains/Chain.ts` for `ChainType`) and `@lifi/data-types` 7.1.6 (`src/chains/supportedChains.*.ts` for each chain's display name, chain type and mainnet flag). These are the latest versions at fetch time. The tarballs are treated as untrusted data. Each is downloaded into memory and its SHA-1 is checked against the registry's `dist.shasum`. Only the named `.ts` members are read, as text, and parsed with regular expressions. Nothing is written to disk or executed.
- `GET /v1/connections?fromChain=1&toChain=<id>&fromToken=0x0…0` for every id that appears in any of the sources above or in the data. Each call records whether the API's request validation accepts the id (HTTP 400 with code 1011 when it does not), the message, and how many destination tokens come back.
- `GET /v1/tools?chainTypes=…STL`, for which bridges list each id in their `supportedChains`. The §4 snapshot kept only the counts.
- `GET /v1/tokens?chains=1201081091099710` with and without `STL`, to explain why §3 found no Stellar tokens.
- `GET /v1/quote` for 100 USDC from Ethereum to Tron (USDT, `toChain=728126428`), to Arbitrum (USDC) and to Blast (ETH). The requests use the example `fromAddress` from LI.FI's API reference and no integrator parameter. For Tron, the receiver is the same 20 bytes in Tron's base58check format. The `BridgeData` tuple is decoded from the returned `transactionRequest.data` to see which `destinationChainId` and `receiver` the Diamond would emit.

Each call writes one line to `lifi_chain_id_registry_sources_raw.jsonl`: the URL, fetch time, HTTP status, and either the source text or a summary of the response. The file is written only when it is missing. `lifi_chain_id_registry.csv` (one row per id) and the chain list with STL are both derived from it, so a re-run reads the files and makes no network call. Two cited web pages, a GitHub issue and LI.FI's changelog, are snapshotted in a separate file after the registry (below), so the registry files are untouched by them.

In [5]:
# 2.1 Chain ids across LI.FI's published sources: one fetch into lifi_chain_id_registry_sources_raw.jsonl, parsers, and the registry built from it.
import hashlib
import io
import re
import tarfile
from urllib.parse import urlencode

ALL_CHAIN_TYPES = "EVM,SVM,UTXO,MVM,TVM,STL"                  # the six values LI.FI's API reference lists for chainTypes; §2 passed the first five
SOURCES_JSONL = REF / "lifi_chain_id_registry_sources_raw.jsonl"
CHAINS_STL_CSV, REGISTRY_CSV = "lifi_api_chains_with_stl_2026-10-28.csv", "lifi_chain_id_registry.csv"
LIFIDATA_PATH = "src/Helpers/LiFiData.sol"
LIFIDATA_SHA = "f5686943150bf16cb0d1de649caef3cfa4dc959c"     # pinned; the fetch records whether it is still the last commit touching the file on main
LIFIDATA_URL = f"https://raw.githubusercontent.com/lifinance/contracts/{LIFIDATA_SHA}/{LIFIDATA_PATH}"
GITHUB_COMMITS_URL = f"https://api.github.com/repos/lifinance/contracts/commits?path={LIFIDATA_PATH}&sha=main&per_page=1"
NPM = {"@lifi/types": ("18.14.0", ["package/src/chains/base.ts", "package/src/chains/Chain.ts"]),
       "@lifi/data-types": ("7.1.6", [f"package/src/chains/supportedChains.{s}.ts" for s in ("evm", "svm", "mvm", "utxo", "tvm", "stl")])}
QUOTE_FROM = "0x552008c0f6870c2f77e5cC1d2eb9bdff03e30Ea0"      # the example fromAddress in LI.FI's API reference (docs.li.fi/openapi.yaml)
USDC_ETHEREUM = "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48"
B58 = "123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz"


def tron_base58(evm_hex):
    "A 20-byte address in Tron's format: base58check of 0x41 followed by the 20 bytes."
    payload = bytes.fromhex("41" + evm_hex.lower().removeprefix("0x"))
    raw = payload + hashlib.sha256(hashlib.sha256(payload).digest()).digest()[:4]
    n, out = int.from_bytes(raw, "big"), ""
    while n:
        n, r = divmod(n, 58)
        out = B58[r] + out
    return out


assert tron_base58("0xa614f803b6fd780986a42c78ec9c7f77e6ded13c") == "TR7NHqjeKQxGTCi8q8ZY4pL8otSzgjLj6t"   # USDT on Tron, a known pair
QUOTE_PROBES = [{"toChain": 728126428, "toToken": "TR7NHqjeKQxGTCi8q8ZY4pL8otSzgjLj6t", "toAddress": tron_base58(QUOTE_FROM)},   # USDT on Tron
                {"toChain": 42161, "toToken": "0xaf88d065e77c8cC2239327C5EDb3A432268e5831", "toAddress": QUOTE_FROM},          # USDC on Arbitrum
                {"toChain": 81457, "toToken": ZERO_ADDRESS, "toAddress": QUOTE_FROM}]                                          # ETH on Blast


def ts_enum(text, name):
    "Members of `export enum <name> { … }` in order: member, value, trailing comment, and the section comment above it."
    body = re.search(rf"export enum {name} \{{\n(.*?)\n\}}", text, re.S).group(1)
    rows, section = [], ""
    for line in body.splitlines():
        s = line.strip()
        if not s:
            continue
        if s.startswith("//"):
            section = s.lstrip("/").strip()
            continue
        m = re.match(r"^(\w+)\s*=\s*'?([^',]+?)'?\s*,?\s*(?://\s*(.*))?$", s)
        rows.append({"member": m.group(1), "value": m.group(2), "comment": m.group(3) or "", "section": section})
    return rows


def lifidata_constants(text):
    non_evm = re.search(r"NON_EVM_ADDRESS\s*=\s*(0x[0-9a-fA-F]{40})", text).group(1)
    ids = [{"constant": f"LIFI_CHAIN_ID_{m.group(1)}", "chain_id": int(m.group(2))} for m in re.finditer(r"LIFI_CHAIN_ID_(\w+)\s*=\s*(\d+);", text)]
    comment = re.search(r"//\s*(LI\.FI non-EVM Custom Chain IDs[^\n]*)", text).group(1)
    return non_evm, ids, comment


def data_types_chains(texts, chain_id_by_member, chain_key_by_member):
    "One row per chain object in @lifi/data-types supportedChains.*.ts, from the fields before `metamask:`."
    rows = []
    for member, text in texts.items():
        starts = [m.start() for m in re.finditer(r"\{\s*\n\s*key: ChainKey\.", text)]
        assert len(starts) == text.count("key: ChainKey."), member
        for i, s in enumerate(starts):
            head = text[s: starts[i + 1] if i + 1 < len(starts) else len(text)].split("metamask:")[0]
            field = lambda pat: (re.search(pat, head) or [None, None])[1]
            id_expr = field(r"\bid: ([\w.]+),")
            cid = int(id_expr) if id_expr.isdigit() else int(chain_id_by_member[id_expr.removeprefix("ChainId.")])
            rows.append({"file": member.rsplit("/", 1)[1], "line": text[:s].count("\n") + 2, "chain_id": cid, "key": chain_key_by_member[field(r"key: ChainKey\.(\w+)")],
                         "chainType": field(r"chainType: ChainType\.(\w+)"), "name": field(r"name: '([^']*)'"), "mainnet": field(r"mainnet: (true|false)") == "true"})
    return rows


def decode_bridge_data(data_hex):
    "BridgeData, the first argument of every LI.FI facet's start function: a dynamic tuple, so the first head word is an offset."
    b = bytes.fromhex(data_hex.removeprefix("0x"))
    args = b[4:]
    word = lambda buf, i: buf[32 * i: 32 * (i + 1)]
    t = args[int.from_bytes(word(args, 0), "big"):]
    string_at = lambda i: (lambda o: t[o + 32: o + 32 + int.from_bytes(t[o: o + 32], "big")].decode())(int.from_bytes(word(t, i), "big"))
    return {"selector": "0x" + b[:4].hex(), "transactionId": "0x" + word(t, 0).hex(), "bridge": string_at(1), "integrator": string_at(2),
            "referrer": "0x" + word(t, 3)[12:].hex(), "sendingAssetId": "0x" + word(t, 4)[12:].hex(), "receiver": "0x" + word(t, 5)[12:].hex(),
            "minAmount": str(int.from_bytes(word(t, 6), "big")), "destinationChainId": int.from_bytes(word(t, 7), "big"),
            "hasSourceSwaps": bool(int.from_bytes(word(t, 8), "big")), "hasDestinationCall": bool(int.from_bytes(word(t, 9), "big"))}


def record(kind, url, status, payload):
    return {"kind": kind, "url": url, "fetched_at_utc": now_utc(), "http_status": status, "payload": payload}


def npm_member_texts(package, version, members):
    "Tarball into memory, SHA-1 checked against the registry's dist.shasum, the named members returned as text. Nothing is extracted to disk or executed."
    status, meta = get_json(f"https://registry.npmjs.org/{package}")
    assert status == 200, (package, status)
    dist = meta["versions"][version]["dist"]
    blob = SESSION.get(dist["tarball"], timeout=120).content
    sha1 = hashlib.sha1(blob).hexdigest()
    assert sha1 == dist["shasum"], (package, version, sha1, dist["shasum"])
    with tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz") as t:
        texts = {m: t.extractfile(t.getmember(m)).read().decode("utf-8") for m in members}
    return dist["tarball"], {"version": version, "latest_at_fetch": meta["dist-tags"]["latest"], "published": meta["time"][version],
                             "shasum": dist["shasum"], "sha1_of_download": sha1, "texts": texts}


def known_ids(recs):
    "Every chain id named by the fetched sources, the first chain snapshot of §2, and the data (source and destination ids)."
    by = {r["kind"]: r for r in recs}
    ids = {c["id"] for c in by["v1_chains_all_types"]["payload"]["chains"]} | set(chains["id"].astype(int))
    types_text = by["npm_@lifi/types"]["payload"]["texts"]["package/src/chains/base.ts"]
    chain_id_enum = ts_enum(types_text, "ChainId")
    ids |= {int(r["value"]) for r in chain_id_enum}
    ids |= {r["chain_id"] for r in lifidata_constants(by["lifidata_sol"]["payload"])[1]}
    ids |= {r["chain_id"] for r in data_types_chains(by["npm_@lifi/data-types"]["payload"]["texts"], {r["member"]: r["value"] for r in chain_id_enum},
                                                       {r["member"]: r["value"] for r in ts_enum(types_text, "ChainKey")})}
    return sorted(ids | set(SOURCE_CHAIN_ID.values()) | {int(i) for i in tf["destinationChainId"].unique()})


def fetch_sources():
    "All calls of this section, keyless, paced at 0.7 s on li.quest (100 requests per minute without a key). Written to a .partial file, renamed when complete."
    partial = SOURCES_JSONL.with_suffix(".partial.jsonl")
    recs = []
    status, body = get_json(f"{LIFI}/chains", params={"chainTypes": ALL_CHAIN_TYPES}, pause=0.7)
    recs.append(record("v1_chains_all_types", f"{LIFI}/chains?chainTypes={ALL_CHAIN_TYPES}", status, body))
    status, body = get_json(f"{LIFI}/chains", params={"chainTypes": "FOO"}, pause=0.7)
    recs.append(record("v1_chains_invalid_type", f"{LIFI}/chains?chainTypes=FOO", status, body))
    status, body = get_json(GITHUB_COMMITS_URL)
    recs.append(record("github_last_commit_lifidata", GITHUB_COMMITS_URL, status,
                       [{"sha": c["sha"], "date": c["commit"]["committer"]["date"], "message": c["commit"]["message"].splitlines()[0]} for c in body] if status == 200 else body))
    r = SESSION.get(LIFIDATA_URL, timeout=60)
    recs.append(record("lifidata_sol", LIFIDATA_URL, r.status_code, r.text))
    for pkg, (ver, members) in NPM.items():
        url, payload = npm_member_texts(pkg, ver, members)
        recs.append(record(f"npm_{pkg}", url, 200, payload))
    status, body = get_json(f"{LIFI}/tools", params={"chainTypes": ALL_CHAIN_TYPES}, pause=0.7)
    recs.append(record("v1_tools", f"{LIFI}/tools?chainTypes={ALL_CHAIN_TYPES}", status,
                       {"bridges": [{"key": b["key"], "name": b["name"], "from": sorted({c["fromChainId"] for c in b.get("supportedChains") or []}),
                                     "to": sorted({c["toChainId"] for c in b.get("supportedChains") or []})} for b in body["bridges"]],
                        "exchanges": [{"key": e["key"], "name": e["name"], "chains": sorted(set(e.get("supportedChains") or []))} for e in body["exchanges"]]}))
    for ct in (CHAIN_TYPES, ALL_CHAIN_TYPES):
        status, body = get_json(f"{LIFI}/tokens", params={"chains": "1201081091099710", "chainTypes": ct}, pause=0.7)
        toks = (body.get("tokens") or {}).get("1201081091099710", []) if status == 200 and isinstance(body, dict) else None
        recs.append(record("v1_tokens_stellar", f"{LIFI}/tokens?chains=1201081091099710&chainTypes={ct}", status,
                           {"chainTypes": ct, "n_tokens": None if toks is None else len(toks), "symbols": sorted(t["symbol"] for t in toks or []), "error": None if toks is not None else body}))
    ids = known_ids(recs)
    print(f"ids to probe on /v1/connections: {len(ids)}")
    for cid in ids:
        status, body = get_json(f"{LIFI}/connections", params={"fromChain": 1, "toChain": cid, "fromToken": ZERO_ADDRESS}, pause=0.7)
        conns = body.get("connections") if status == 200 and isinstance(body, dict) else None
        recs.append(record("v1_connections", f"{LIFI}/connections?fromChain=1&toChain={cid}&fromToken={ZERO_ADDRESS}", status,
                           {"toChain": cid, "message": body.get("message") if isinstance(body, dict) else str(body)[:300], "code": body.get("code") if isinstance(body, dict) else None,
                            "n_connections": None if conns is None else len(conns), "n_to_tokens": None if conns is None else sum(len(c.get("toTokens") or []) for c in conns)}))
    for p in QUOTE_PROBES:
        params = {"fromChain": 1, "toChain": p["toChain"], "fromToken": USDC_ETHEREUM, "toToken": p["toToken"], "fromAmount": "100000000", "fromAddress": QUOTE_FROM, "toAddress": p["toAddress"]}
        status, body = get_json(f"{LIFI}/quote", params=params, pause=0.7)
        payload = {"params": params}
        if status == 200:
            tr = body.get("transactionRequest") or {}
            payload.update({"tool": body.get("tool"), "action_fromChainId": body["action"]["fromChainId"], "action_toChainId": body["action"]["toChainId"],
                            "transactionRequest_to": tr.get("to"), "transactionRequest_data": tr.get("data")})
        else:
            payload["error"] = body
        recs.append(record("v1_quote", f"{LIFI}/quote?{urlencode(params)}", status, payload))
    with partial.open("w") as fh:
        for rec in recs:
            fh.write(json.dumps(rec) + "\n")
    partial.rename(SOURCES_JSONL)
    return recs


if SOURCES_JSONL.exists():
    sources = [json.loads(line) for line in SOURCES_JSONL.read_text().splitlines()]
    print(f"{SOURCES_JSONL.name}: read snapshot, {len(sources)} calls, fetched {sources[0]['fetched_at_utc']} to {sources[-1]['fetched_at_utc']} (no network call)")
else:
    print(f"{SOURCES_JSONL.name}: no snapshot, fetching")
    sources = fetch_sources()
    manifest_write(SOURCES_JSONL.name, f"{LIFI}/chains, /tools, /tokens, /connections, /quote; {GITHUB_COMMITS_URL}; {LIFIDATA_URL}; "
                   + "; ".join(f"https://registry.npmjs.org/{p}/-/{p.split('/')[1]}-{v}.tgz" for p, (v, _) in NPM.items()),
                   len(sources), "01 §2.1", sources[0]["fetched_at_utc"],
                   note=f"{len(sources)} calls from {sources[0]['fetched_at_utc']} to {sources[-1]['fetched_at_utc']}, no API key; "
                        f"by kind {pd.Series([r['kind'] for r in sources]).value_counts().to_dict()}; HTTP statuses {pd.Series([r['http_status'] for r in sources]).value_counts().to_dict()}; "
                        "npm tarballs read in memory after a SHA-1 check against dist.shasum, nothing extracted or executed")
    print(f"{SOURCES_JSONL.name}: {len(sources)} calls written")
SRC = {}
for r in sources:
    SRC.setdefault(r["kind"], []).append(r)
print("calls by kind:", {k: len(v) for k, v in SRC.items()}, " HTTP statuses:", pd.Series([r["http_status"] for r in sources]).value_counts().to_dict())

lifi_chain_id_registry_sources_raw.jsonl: read snapshot, 107 calls, fetched 2026-10-28T21:35:06+00:00 to 2026-10-28T21:36:58+00:00 (no network call)
calls by kind: {'v1_chains_all_types': 1, 'v1_chains_invalid_type': 1, 'github_last_commit_lifidata': 1, 'lifidata_sol': 1, 'npm_@lifi/types': 1, 'npm_@lifi/data-types': 1, 'v1_tools': 1, 'v1_tokens_stellar': 2, 'v1_connections': 95, 'v1_quote': 3}  HTTP statuses: {200: 83, 400: 24}


In [6]:
# The sources parsed, the chain list with STL and the registry derived from the JSONL (no network call), and what each source says.
one = lambda kind: SRC[kind][0]["payload"]
types_text = one("npm_@lifi/types")["texts"]["package/src/chains/base.ts"]
chain_id_enum = pd.DataFrame(ts_enum(types_text, "ChainId"))
chain_key_enum = pd.DataFrame(ts_enum(types_text, "ChainKey"))
chain_type_enum = [r["value"] for r in ts_enum(one("npm_@lifi/types")["texts"]["package/src/chains/Chain.ts"], "ChainType")]
chain_id_enum["chain_id"] = chain_id_enum["value"].astype("int64")
chain_id_enum["key"] = chain_id_enum["member"].map(dict(zip(chain_key_enum["member"], chain_key_enum["value"])))
dtc = pd.DataFrame(data_types_chains(one("npm_@lifi/data-types")["texts"], dict(zip(chain_id_enum["member"], chain_id_enum["value"])), dict(zip(chain_key_enum["member"], chain_key_enum["value"]))))
NON_EVM_ADDRESS, lifidata_ids, lifidata_comment = lifidata_constants(one("lifidata_sol"))
lifidata = pd.DataFrame(lifidata_ids)
assert NON_EVM_ADDRESS.lower() == SENTINEL_RECEIVER


def chain_row(c):
    nt = c.get("nativeToken") or {}
    return {"id": c["id"], "key": c["key"], "name": c["name"], "chainType": c["chainType"], "coin": c.get("coin"), "mainnet": c.get("mainnet"),
            "native_symbol": nt.get("symbol"), "native_name": nt.get("name"), "native_decimals": nt.get("decimals"), "native_address": nt.get("address"),
            "diamondAddress": c.get("diamondAddress")}


def derived_csv(name, build, source_url, note):
    "Write data/reference/<name> from the JSONL when it is missing (no network call); the manifest takes the JSONL's fetch time."
    path = REF / name
    if path.exists():
        print(f"{name}: read snapshot, derived from {SOURCES_JSONL.name} (no network call)")
    else:
        df = build()
        df.to_csv(path, index=False)
        manifest_write(name, source_url, len(df), "01 §2.1", sources[0]["fetched_at_utc"], note=f"{note}; derived from {SOURCES_JSONL.name} at {now_utc()}")
        print(f"{name}: written from {SOURCES_JSONL.name}, {len(df)} rows")
    return path


derived_csv(CHAINS_STL_CSV, lambda: pd.DataFrame([chain_row(c) for c in one("v1_chains_all_types")["chains"]]).sort_values("id").reset_index(drop=True), SRC["v1_chains_all_types"][0]["url"],
            "chainTypes with all six values LI.FI's API reference lists (EVM, SVM, UTXO, MVM, TVM, STL); lifi_api_chains.csv (§2) passed five and lacks Stellar")
chains_stl = pd.read_csv(REF / CHAINS_STL_CSV, dtype={"key": str, "native_address": str, "diamondAddress": str, "native_symbol": str, "coin": str})
assert list(chains_stl.columns) == list(chains.columns)

conn = pd.DataFrame([r["payload"] | {"http_status": r["http_status"]} for r in SRC["v1_connections"]]).rename(columns={"toChain": "chain_id"})
tools_live = one("v1_tools")
quotes = []
for r in SRC["v1_quote"]:
    p = r["payload"]
    q = {"toChain": p["params"]["toChain"], "toToken": p["params"]["toToken"], "toAddress": p["params"]["toAddress"], "http_status": r["http_status"]}
    if r["http_status"] == 200:
        q |= {"tool": p["tool"], "action_toChainId": p["action_toChainId"], "transactionRequest_to": p["transactionRequest_to"]} | {f"bridgeData.{k}": v for k, v in decode_bridge_data(p["transactionRequest_data"]).items()}
    else:
        q |= {"error": p["error"]}
    quotes.append(q)
quotes = pd.DataFrame(quotes)
for c in ("action_toChainId", "bridgeData.destinationChainId"):
    quotes[c] = quotes[c].astype("Int64")


# Pairing a LiFiData.sol constant with an API id: the same id when an API source carries it, else the constant's suffix equal to a chain name or a ChainId member (case-insensitive).
api_name = dict(zip(chains_stl["id"], chains_stl["name"])) | {k: v for k, v in zip(dtc["chain_id"], dtc["name"]) if k not in set(chains_stl["id"])}
by_name = {n.lower(): i for i, n in api_name.items()} | {m.lower(): i for m, i in zip(chain_id_enum["member"], chain_id_enum["chain_id"])}
api_ids = set(chains_stl["id"]) | set(chain_id_enum["chain_id"]) | set(dtc["chain_id"])
lifidata["api_chain_id"] = pd.array([cid if cid in api_ids else by_name.get(c.removeprefix("LIFI_CHAIN_ID_").lower()) for c, cid in zip(lifidata["constant"], lifidata["chain_id"])], dtype="Int64")
lifidata["paired_by"] = ["same id" if cid in api_ids else ("constant name" if pd.notna(a) else None) for cid, a in zip(lifidata["chain_id"], lifidata["api_chain_id"])]
contract_of = {int(a): (c, cid) for c, cid, a in zip(lifidata["constant"], lifidata["chain_id"], lifidata["api_chain_id"]) if pd.notna(a) and a != cid}


REG_INT = ["api_chain_id", "connections_code", "connections_to_tokens", "quote_onchain_destination_chain_id"]


def build_registry():
    dest_ids = {int(i) for i in tf["destinationChainId"].unique()}
    rows = []
    for cid in known_ids(sources):
        v1 = chains_stl[chains_stl["id"] == cid]
        dt = dtc[dtc["chain_id"] == cid]
        en = chain_id_enum[chain_id_enum["chain_id"] == cid]
        ld = lifidata[lifidata["chain_id"] == cid]
        cn = conn[conn["chain_id"] == cid]
        in_v1, in_dt, in_en, in_ld = len(v1) > 0, len(dt) > 0, len(en) > 0, len(ld) > 0
        in_api = in_v1 or in_dt or in_en
        paired_api = int(ld["api_chain_id"].iloc[0]) if in_ld and ld["api_chain_id"].notna().iloc[0] else None
        if in_v1:
            name, key, ctype, mainnet, name_source = v1["name"].iloc[0], v1["key"].iloc[0], v1["chainType"].iloc[0], bool(v1["mainnet"].iloc[0]), "/v1/chains"
        elif in_dt:
            name, key, ctype, mainnet, name_source = dt["name"].iloc[0], dt["key"].iloc[0], dt["chainType"].iloc[0], bool(dt["mainnet"].iloc[0]), "@lifi/data-types supportedChains"
        elif in_en:   # no name in any source; the type only where the member sits in ChainKey's "EVM" section
            ck_section = chain_key_enum.loc[chain_key_enum["value"] == en["key"].iloc[0], "section"]
            ctype = "EVM" if len(ck_section) and ck_section.iloc[0] == "EVM" else None
            name, key, mainnet, name_source = None, en["key"].iloc[0], en["section"].iloc[0] != "Testnet", "@lifi/types ChainKey (no name; type from its ChainKey section)"
        else:
            pv = chains_stl[chains_stl["id"] == paired_api] if paired_api else chains_stl.iloc[0:0]
            name, key, ctype, mainnet = (pv["name"].iloc[0], pv["key"].iloc[0], pv["chainType"].iloc[0], bool(pv["mainnet"].iloc[0])) if len(pv) else (ld["constant"].iloc[0].removeprefix("LIFI_CHAIN_ID_"), None, None, None)
            name_source = f"/v1/chains name of {paired_api}, paired by {ld['constant'].iloc[0]}" if len(pv) else f"LiFiData.sol constant {ld['constant'].iloc[0]}"
        evm, seen = ctype == "EVM", cid in dest_ids
        if in_api and in_ld:
            namespace, basis = "API and contract", f"{ld['constant'].iloc[0]} in LiFiData.sol equals the API id"
        elif in_api and evm:
            namespace, basis = "API and contract", "EVM chain: its EIP-155 id, used on-chain and in the API"
        elif in_api and seen:
            namespace, basis = "API and contract", "non-EVM id seen as destinationChainId in transfers_started.csv; no LiFiData.sol constant has this value"
        elif in_api:
            namespace, basis = "API only", "non-EVM id in the API's sources; no LiFiData.sol constant has this value and the data does not show it on-chain"
        else:
            namespace, basis = "contract only", f"{ld['constant'].iloc[0]} in LiFiData.sol; " + (f"the API's id for the same chain is {paired_api}" if paired_api else "no API id for this chain")
        if cid in contract_of:
            basis += f"; LiFiData.sol's {contract_of[cid][0]} for this chain is {contract_of[cid][1]}"
        q = quotes[(quotes["toChain"] == cid) & (quotes["http_status"] == 200)]
        tb_to = sorted(b["key"] for b in tools_live["bridges"] if cid in b["to"])
        tb_from = sorted(b["key"] for b in tools_live["bridges"] if cid in b["from"])
        tx = sorted(e["key"] for e in tools_live["exchanges"] if cid in e["chains"])
        srcs = [s for s, ok in [("/v1/chains (all six chainTypes)", in_v1), ("lifi_api_chains.csv (§2, five chainTypes)", cid in set(chains["id"])),
                                ("@lifi/types 18.14.0 ChainId", in_en), ("@lifi/data-types 7.1.6 supportedChains", in_dt), (f"LiFiData.sol @{LIFIDATA_SHA[:8]}", in_ld),
                                ("transfers_started.csv destinationChainId", seen)] if ok]
        rows.append({"chain_id": cid, "name": name, "key": key, "chain_type": ctype, "mainnet": mainnet, "name_source": name_source, "namespace": namespace, "namespace_basis": basis,
                     "api_chain_id": cid if in_api else paired_api, "listed_by_v1_chains": in_v1, "listed_in_lifi_api_chains_csv": cid in set(chains["id"]),
                     "types_chainid_member": en["member"].iloc[0] if in_en else None, "types_chainid_section": en["section"].iloc[0] if in_en else None,
                     "types_chainid_comment": (en["comment"].iloc[0] or None) if in_en else None, "data_types_name": dt["name"].iloc[0] if in_dt else None,
                     "data_types_chain_type": dt["chainType"].iloc[0] if in_dt else None, "lifidata_constant": ld["constant"].iloc[0] if in_ld else None,
                     "seen_in_export_as_destination": seen,
                     "connections_http_status": int(cn["http_status"].iloc[0]), "connections_code": cn["code"].iloc[0] if pd.notna(cn["code"].iloc[0]) else None,
                     "connections_message": cn["message"].iloc[0], "connections_to_tokens": cn["n_to_tokens"].iloc[0],
                     "tools_bridges_to": "|".join(tb_to) or None, "tools_bridges_from": "|".join(tb_from) or None, "tools_exchanges": "|".join(tx) or None,
                     "quote_tool": q["tool"].iloc[0] if len(q) else None, "quote_onchain_destination_chain_id": q["bridgeData.destinationChainId"].iloc[0] if len(q) else None,
                     "quote_onchain_receiver": q["bridgeData.receiver"].iloc[0] if len(q) else None, "sources": "|".join(srcs)})
    df = pd.DataFrame(rows)
    for c in REG_INT:   # nullable integers, so the CSV holds 728126428 and not 728126428.0
        df[c] = pd.array([None if pd.isna(r[c]) else int(r[c]) for r in rows], dtype="Int64")
    return df


derived_csv(REGISTRY_CSV, build_registry, f"the calls in {SOURCES_JSONL.name}: {LIFI}/chains, /connections, /tools, /quote; {LIFIDATA_URL}; @lifi/types 18.14.0 and @lifi/data-types 7.1.6 from registry.npmjs.org",
            "one row per chain id named by any source or by the data: name, key, chain type, LI.FI namespace (API, contract or both) with its basis, membership in each source, "
            "the /v1/connections validation probe, the bridges /v1/tools lists, and the destinationChainId a /v1/quote calldata carries")
registry =pd.read_csv(REF / REGISTRY_CSV, dtype={c: "Int64" for c in REG_INT} | {"key": str, "types_chainid_member": str})
print(f"{REGISTRY_CSV}: {len(registry)} ids; by namespace {registry['namespace'].value_counts().to_dict()}; by chain type {registry['chain_type'].value_counts(dropna=False).to_dict()}")

lifi_api_chains_with_stl_2026-10-28.csv: read snapshot, derived from lifi_chain_id_registry_sources_raw.jsonl (no network call)
lifi_chain_id_registry.csv: read snapshot, derived from lifi_chain_id_registry_sources_raw.jsonl (no network call)
lifi_chain_id_registry.csv: 95 ids; by namespace {'API and contract': 90, 'API only': 3, 'contract only': 2}; by chain type {'EVM': 81, 'UTXO': 5, nan: 3, 'TVM': 2, 'SVM': 2, 'STL': 1, 'MVM': 1}


In [7]:
# What each source says (reference data).
gh = one("github_last_commit_lifidata")
print(f"LiFiData.sol pinned at {LIFIDATA_SHA}; last commit touching it on main at fetch time: {gh[0]['sha']} ({gh[0]['date']}, \"{gh[0]['message']}\"); the pinned commit is the last: {gh[0]['sha'] == LIFIDATA_SHA}")
print(f"  NON_EVM_ADDRESS = {NON_EVM_ADDRESS}, the data's receiver sentinel in checksummed case; comment above the ids: \"{lifidata_comment}\"")
display(lifidata.merge(registry[["chain_id", "name", "key", "chain_type", "namespace", "listed_by_v1_chains", "seen_in_export_as_destination"]], on="chain_id", how="left"))
for pkg in NPM:
    m = one(f"npm_{pkg}")
    print(f"{pkg} {m['version']} (published {m['published']}; latest at fetch {m['latest_at_fetch']}); SHA-1 of the download {m['sha1_of_download']} equals dist.shasum: {m['sha1_of_download'] == m['shasum']}")
print("ChainType enum in @lifi/types:", chain_type_enum)
print(f"@lifi/types ChainId: {len(chain_id_enum)} members, by section {chain_id_enum['section'].replace('', '(no section comment: EVM)').value_counts().to_dict()}")
print(f"@lifi/data-types supportedChains: {len(dtc)} chains, by type {dtc['chainType'].value_counts().to_dict()}, mainnet {int(dtc['mainnet'].sum())}")
print("ChainId members whose line carries a comment, and the section comments of the non-EVM ids:")
display(chain_id_enum[(chain_id_enum["comment"] != "") | (chain_id_enum["section"] != "")][["member", "chain_id", "key", "section", "comment"]])

print(f"/v1/chains: §2 snapshot {len(chains)} chains {chains['chainType'].value_counts().to_dict()}; with all six chainTypes {len(chains_stl)} chains {chains_stl['chainType'].value_counts().to_dict()}")
print("  ids added by STL:", chains_stl.loc[~chains_stl["id"].isin(chains["id"]), ["id", "key", "name", "chainType"]].to_dict("records"),
      " ids in §2 but not in the new list:", sorted(set(chains["id"]) - set(chains_stl["id"])))
print("  name, key or type changed between the two lists for the same id:",
      int((chains.set_index("id")[["name", "key", "chainType"]] != chains_stl.set_index("id").loc[chains["id"], ["name", "key", "chainType"]]).any(axis=1).sum()))
print("/v1/chains with chainTypes=FOO:", SRC["v1_chains_invalid_type"][0]["http_status"], SRC["v1_chains_invalid_type"][0]["payload"])
for r in SRC["v1_tokens_stellar"]:
    print(f"/v1/tokens?chains=1201081091099710&chainTypes={r['payload']['chainTypes']}: HTTP {r['http_status']}, {r['payload']['n_tokens']} tokens {r['payload']['symbols'][:8]}")

print("\nwhich sources carry which ids:")
src_sets = {"/v1/chains (six chainTypes)": set(chains_stl["id"]), "@lifi/types ChainId": set(chain_id_enum["chain_id"]), "@lifi/data-types supportedChains": set(dtc["chain_id"]),
            "LiFiData.sol": set(lifidata["chain_id"]), "data destinationChainId": set(dest_ids_all := {int(i) for i in tf["destinationChainId"].unique()})}
display(pd.DataFrame([{"source": a, **{f"ids also in {b}": len(s & t) for b, t in src_sets.items()}} for a, s in src_sets.items()]).set_index("source"))   # the diagonal is each source's own count
enum_unlisted = registry[registry["types_chainid_member"].notna() & ~registry["listed_by_v1_chains"]]
print(f"ids in @lifi/types ChainId but not in /v1/chains: {len(enum_unlisted)}; of them in @lifi/data-types supportedChains: {int(enum_unlisted['data_types_name'].notna().sum())}, "
      f"all with mainnet true: {bool(enum_unlisted.loc[enum_unlisted['data_types_name'].notna(), 'mainnet'].astype(bool).all())}; accepted by /v1/connections: {enum_unlisted.loc[enum_unlisted['connections_http_status'] == 200, 'data_types_name'].tolist()}")
print(enum_unlisted[["chain_id", "types_chainid_member", "data_types_name", "types_chainid_section", "connections_http_status"]].to_dict("records"))
print("ids in /v1/chains but not in @lifi/types ChainId:", sorted(set(chains_stl["id"]) - set(chain_id_enum["chain_id"])))

print("\n/v1/connections validation (toChain=<id> from Ethereum's native token) against /v1/chains membership, all ids in the registry:")
display(pd.crosstab(registry["listed_by_v1_chains"], registry["connections_http_status"], margins=True))
print("messages:", registry.groupby(["connections_http_status", "connections_code"], dropna=False)["connections_message"].agg(lambda s: sorted(set(s.dropna()))[:2]).to_dict())

print("\n/v1/quote, 100 USDC from Ethereum, BridgeData decoded from transactionRequest.data (no integrator parameter sent):")
display(quotes.T)
print("\nthe registry, ids /v1/chains does not list:")
display(registry[~registry["listed_by_v1_chains"]][["chain_id", "name", "key", "chain_type", "mainnet", "name_source", "namespace", "api_chain_id", "seen_in_export_as_destination",
                                                     "connections_http_status", "tools_bridges_to", "sources"]])
reg_i = registry.set_index("chain_id")
print("bridges /v1/tools lists with Tron (728126428) or Stellar as a destination:", reg_i.loc[[728126428, 1201081091099710], "tools_bridges_to"].to_dict())
print("ids /v1/tools lists for any bridge, among 81457, 204, 60808, 232 and 1885080386571452:",
      reg_i.loc[[81457, 204, 60808, 232, 1885080386571452], ["tools_bridges_to", "tools_bridges_from", "tools_exchanges"]].dropna(how="all").index.tolist())

LiFiData.sol pinned at f5686943150bf16cb0d1de649caef3cfa4dc959c; last commit touching it on main at fetch time: f5686943150bf16cb0d1de649caef3cfa4dc959c (2026-07-16T13:49:00Z, "Polymer hypercore ex-286 [PolymerCCTPFacet v3.0.0, ITokenMessenger v1.1.0] (#1652)"); the pinned commit is the last: True
  NON_EVM_ADDRESS = 0x11f111f111f111F111f111f111F111f111f111F1, the data's receiver sentinel in checksummed case; comment above the ids: "LI.FI non-EVM Custom Chain IDs (IDs are made up by the LI.FI team)"


,constant,chain_id,api_chain_id,paired_by,name,key,chain_type,namespace,listed_by_v1_chains,seen_in_export_as_destination
0,LIFI_CHAIN_ID_APTOS,9271000000000010,<NA>,NaN,APTOS,NaN,NaN,contract only,False,False
1,LIFI_CHAIN_ID_BCH,20000000000002,20000000000002,same id,Bitcoin Cash,bch,UTXO,API and contract,False,False
2,LIFI_CHAIN_ID_BTC,20000000000001,20000000000001,same id,Bitcoin,btc,UTXO,API and contract,True,True
3,LIFI_CHAIN_ID_DGE,20000000000004,20000000000004,same id,Dogecoin,dge,UTXO,API and contract,False,False
4,LIFI_CHAIN_ID_LTC,20000000000003,20000000000003,same id,Litecoin,ltc,UTXO,API and contract,False,False
5,LIFI_CHAIN_ID_SOLANA,1151111081099710,1151111081099710,same id,Solana,sol,SVM,API and contract,True,True
6,LIFI_CHAIN_ID_STELLAR,1201081091099710,1201081091099710,same id,Stellar,xlm,STL,API and contract,True,True
7,LIFI_CHAIN_ID_SUI,9270000000000000,9270000000000000,same id,Sui,sui,MVM,API and contract,True,True
8,LIFI_CHAIN_ID_TRON,1885080386571452,728126428,constant name,Tron,trn,TVM,contract only,False,True
9,LIFI_CHAIN_ID_HYPERCORE,1337,1337,same id,Hyperliquid,hpl,EVM,API and contract,True,True


@lifi/types 18.14.0 (published 2026-10-20T15:52:23.973Z; latest at fetch 18.14.0); SHA-1 of the download c89100a04f66cbb7ef90f4e2771260a815e198e6 equals dist.shasum: True
@lifi/data-types 7.1.6 (published 2026-09-25T21:34:26.629Z; latest at fetch 7.1.6); SHA-1 of the download 15edcc95a8d871bae2bbce4236167dba765ed2e2 equals dist.shasum: True
ChainType enum in @lifi/types: ['EVM', 'SVM', 'MVM', 'UTXO', 'TVM', 'STL']
@lifi/types ChainId: 93 members, by section {'(no section comment: EVM)': 76, 'UTXO (IDs are made up by the LI.FI team)': 5, 'Testnet': 5, 'None-EVM (IDs are made up by the LI.FI team)': 2, 'SVM': 2, 'MVM (IDs are made up by the LI.FI team)': 1, 'Tron': 1, 'Stellar (ID pattern: ASCII decimal of lowercase key letters + 1099710 suffix, x=120, l=108, m=109)': 1}
@lifi/data-types supportedChains: 90 chains, by type {'EVM': 80, 'UTXO': 5, 'SVM': 2, 'MVM': 1, 'TVM': 1, 'STL': 1}, mainnet 85
ChainId members whose line carries a comment, and the section comments of the non-EVM ids:


,member,chain_id,key,section,comment
51,BOC,60808,boc,,BOB was already taken by Boba
59,HPL,1337,hpl,,Hyperliquid's core (non-EVM) chain hard-codes 1337 in ev...
76,TER,1161011141099710,ter,None-EVM (IDs are made up by the LI.FI team),
77,OAS,111971151099710,oas,None-EVM (IDs are made up by the LI.FI team),
78,SOL,1151111081099710,sol,SVM,
79,FOG,1021111031099710,fog,SVM,
80,SUI,9270000000000000,sui,MVM (IDs are made up by the LI.FI team),First 16 non-letter hex digits of SUI genesis blob
81,BTC,20000000000001,btc,UTXO (IDs are made up by the LI.FI team),
82,BCH,20000000000002,bch,UTXO (IDs are made up by the LI.FI team),
83,LTC,20000000000003,ltc,UTXO (IDs are made up by the LI.FI team),


/v1/chains: §2 snapshot 70 chains {'EVM': 65, 'UTXO': 2, 'TVM': 1, 'SVM': 1, 'MVM': 1}; with all six chainTypes 71 chains {'EVM': 65, 'UTXO': 2, 'TVM': 1, 'SVM': 1, 'STL': 1, 'MVM': 1}
  ids added by STL: [{'id': 1201081091099710, 'key': 'xlm', 'name': 'Stellar', 'chainType': 'STL'}]  ids in §2 but not in the new list: []
  name, key or type changed between the two lists for the same id: 0
/v1/chains with chainTypes=FOO: 400 {'message': '/chainTypes/0 must be equal to one of the allowed values', 'code': 1011}
/v1/tokens?chains=1201081091099710&chainTypes=EVM,SVM,UTXO,MVM,TVM: HTTP 200, 0 tokens []
/v1/tokens?chains=1201081091099710&chainTypes=EVM,SVM,UTXO,MVM,TVM,STL: HTTP 200, 20 tokens ['ACT', 'AFR', 'AQUA', 'BLND', 'CETES', 'DAWG', 'EURC', 'LIBRE']

which sources carry which ids:


,ids also in /v1/chains (six chainTypes),ids also in @lifi/types ChainId,ids also in @lifi/data-types supportedChains,ids also in LiFiData.sol,ids also in data destinationChainId
source,,,,,
/v1/chains (six chainTypes),71,71,71,5,64
@lifi/types ChainId,71,93,90,8,68
@lifi/data-types supportedChains,71,90,90,8,68
LiFiData.sol,5,8,8,10,6
data destinationChainId,64,68,68,6,69


... (3 outputs left out)

,0,1,2
toChain,728126428,42161,81457
toToken,TR7NHqjeKQxGTCi8q8ZY4pL8otSzgjLj6t,0xaf88d065e77c8cC2239327C5EDb3A432268e5831,0x0000000000000000000000000000000000000000
toAddress,THjJp93JMSymPZLxvxCqmebBhbHHmwYGUq,0x552008c0f6870c2f77e5cC1d2eb9bdff03e30Ea0,0x552008c0f6870c2f77e5cC1d2eb9bdff03e30Ea0
http_status,200,200,400
tool,near,across,NaN
action_toChainId,728126428,42161,<NA>
transactionRequest_to,0x1231DEB6f5749EF6cE6943a275A1D3E7486F4EaE,0x1231DEB6f5749EF6cE6943a275A1D3E7486F4EaE,NaN
bridgeData.selector,0x02631b09,0x1794958f,NaN
bridgeData.transactionId,0x75acfff22c57051a8896ffd1a1bf62c1e776a28a8f4f7dcbccec26...,0xa30b6a79071d3484a54b93f1c7f6de3f7c57a1c1939cc3cc0a9fe3...,NaN
bridgeData.bridge,near,across,NaN



the registry, ids /v1/chains does not list:


,chain_id,name,key,chain_type,mainnet,name_source,namespace,api_chain_id,seen_in_export_as_destination,connections_http_status,tools_bridges_to,sources
10,106,Velas,vel,EVM,True,@lifi/data-types supportedChains,API and contract,106,False,400,NaN,@lifi/types 18.14.0 ChainId|@lifi/data-types 7.1.6 suppo...
17,204,opBNB,opb,EVM,True,@lifi/data-types supportedChains,API and contract,204,True,400,NaN,@lifi/types 18.14.0 ChainId|@lifi/data-types 7.1.6 suppo...
18,232,Lens,lns,EVM,True,@lifi/data-types supportedChains,API and contract,232,True,400,NaN,@lifi/types 18.14.0 ChainId|@lifi/data-types 7.1.6 suppo...
19,250,Fantom,ftm,EVM,True,@lifi/data-types supportedChains,API and contract,250,False,400,NaN,@lifi/types 18.14.0 ChainId|@lifi/data-types 7.1.6 suppo...
21,288,Boba,bob,EVM,True,@lifi/data-types supportedChains,API and contract,288,False,400,NaN,@lifi/types 18.14.0 ChainId|@lifi/data-types 7.1.6 suppo...
28,1101,Polygon zkEVM,pze,EVM,True,@lifi/data-types supportedChains,API and contract,1101,False,400,NaN,@lifi/types 18.14.0 ChainId|@lifi/data-types 7.1.6 suppo...
30,1284,Moonbeam,moo,EVM,True,@lifi/data-types supportedChains,API and contract,1284,False,400,NaN,@lifi/types 18.14.0 ChainId|@lifi/data-types 7.1.6 suppo...
31,1285,Moonriver,mor,EVM,True,@lifi/data-types supportedChains,API and contract,1285,False,400,NaN,@lifi/types 18.14.0 ChainId|@lifi/data-types 7.1.6 suppo...
48,6900,Nibiru,nib,EVM,True,@lifi/data-types supportedChains,API and contract,6900,False,400,NaN,@lifi/types 18.14.0 ChainId|@lifi/data-types 7.1.6 suppo...
51,9001,Evmos,evm,EVM,True,@lifi/data-types supportedChains,API and contract,9001,False,400,NaN,@lifi/types 18.14.0 ChainId|@lifi/data-types 7.1.6 suppo...


bridges /v1/tools lists with Tron (728126428) or Stellar as a destination: {728126428: 'allbridge|eco|gasZipBridge|layerswap|lifiIntents|near|smartDeposits|symbiosis', 1201081091099710: 'layerswap|near|polymer|polymerStandard'}
ids /v1/tools lists for any bridge, among 81457, 204, 60808, 232 and 1885080386571452: []


**Pages cited.** Two web pages are snapshotted here, fetched once, without a key, into `lifi_changelog_and_vaultpilot_issue_799_raw.jsonl`, one line per call:

- **The GitHub issue**, `agenthill/vaultpilot-mcp#799`, through the GitHub API. The line keeps its number, title, author, state, dates, URL and full body. It is not a LI.FI source.
- **LI.FI's changelog.** `https://docs.li.fi/changelog` redirects to `/changelog/monthly-updates`, the page read for the chain ids. That page is rendered by JavaScript, so its line keeps only the final URL, size and SHA-256. The same page as markdown, `/changelog/monthly-updates.md`, is kept in full.



In [8]:
# 2.1 (pages cited): the GitHub issue and LI.FI's monthly changelog, fetched once into a JSONL; read from it on every later run.
PAGES_JSONL = REF / "lifi_changelog_and_vaultpilot_issue_799_raw.jsonl"
ISSUE_URL = "https://api.github.com/repos/agenthill/vaultpilot-mcp/issues/799"
CHANGELOG_URL = "https://docs.li.fi/changelog"                          # redirects to /changelog/monthly-updates
CHANGELOG_MD_URL = "https://docs.li.fi/changelog/monthly-updates.md"   # the same page as markdown


def fetch_pages():
    partial = PAGES_JSONL.with_suffix(".partial.jsonl")
    recs = []
    status, body = get_json(ISSUE_URL)
    keep = ["number", "title", "state", "created_at", "closed_at", "html_url", "body"]
    recs.append(record("github_issue", ISSUE_URL, status, ({k: body.get(k) for k in keep} | {"user": (body.get("user") or {}).get("login")}) if status == 200 else body))
    r = SESSION.get(CHANGELOG_URL, timeout=60)
    recs.append(record("lifi_changelog_html", CHANGELOG_URL, r.status_code, {"final_url": r.url, "bytes": len(r.content), "sha256": hashlib.sha256(r.content).hexdigest()}))
    r = SESSION.get(CHANGELOG_MD_URL, timeout=60)
    recs.append(record("lifi_changelog_markdown", CHANGELOG_MD_URL, r.status_code, {"bytes": len(r.content), "sha256": hashlib.sha256(r.content).hexdigest(), "text": r.text}))
    with partial.open("w") as fh:
        for rec in recs:
            fh.write(json.dumps(rec) + "\n")
    partial.rename(PAGES_JSONL)
    return recs


if PAGES_JSONL.exists():
    pages = [json.loads(line) for line in PAGES_JSONL.read_text().splitlines()]
    print(f"{PAGES_JSONL.name}: read snapshot, {len(pages)} calls, fetched {pages[0]['fetched_at_utc']} to {pages[-1]['fetched_at_utc']} (no network call)")
else:
    print(f"{PAGES_JSONL.name}: no snapshot, fetching")
    pages = fetch_pages()
    sizes = {p["kind"]: p["payload"].get("bytes") for p in pages if isinstance(p["payload"], dict) and "bytes" in p["payload"]}
    manifest_write(PAGES_JSONL.name, f"{ISSUE_URL}; {CHANGELOG_URL}; {CHANGELOG_MD_URL}", len(pages), "01 §2.1", pages[0]["fetched_at_utc"],
                   note=f"{len(pages)} calls from {pages[0]['fetched_at_utc']} to {pages[-1]['fetched_at_utc']}, no key; HTTP statuses {[p['http_status'] for p in pages]}; "
                        f"issue fields kept: number, title, user, state, created_at, closed_at, html_url, body; changelog page sizes in bytes {sizes} (HTML: final URL, size and SHA-256 only; markdown: full text)")
    print(f"{PAGES_JSONL.name}: {len(pages)} calls written")
PG = {p["kind"]: p for p in pages}

issue = PG["github_issue"]["payload"]
print(f"\nissue #{issue['number']} \"{issue['title'][:90]}…\" by {issue['user']}, created {issue['created_at']}, state {issue['state']}, closed {issue['closed_at']}; {issue['html_url']}")
print("issue body lines that name a chain id, /v1/chains or a chain count:")
for line in issue["body"].splitlines():
    if re.search(r"1885080386571452|728126428|v1/chains|\b\d+ chains\b|chainTypes", line):
        print("  |", line.strip()[:220])

html, md = PG["lifi_changelog_html"], PG["lifi_changelog_markdown"]
print(f"\nchangelog: {html['url']} -> {html['payload']['final_url']} (HTTP {html['http_status']}, {html['payload']['bytes']:,} bytes HTML); "
      f"markdown {md['url']} HTTP {md['http_status']}, {md['payload']['bytes']:,} bytes")
text = md["payload"]["text"]
months = re.findall(r"^## (.+)$", text, re.M)
print(f"months covered: {len(months)}, from {months[-1]} to {months[0]}")
hits = [(i + 1, line.strip()) for i, line in enumerate(text.splitlines()) if re.search(r"\b(blast|opbnb|bob|lens)\b", line, re.I)]
print("lines naming Blast, opBNB, BOB or Lens (case-insensitive, whole word):", hits if hits else "none")
month_of = lambda pos: (re.findall(r"^## (.+)$", text[:pos], re.M) or ["(before the first month)"])[-1]
print("lines with 'inactive' or a chain removal, with their month:")
for m in re.finditer(r"^.*\b(inactive|[Cc]hains? removed|[Rr]emoved .{0,30}chains?)\b.*$", text, re.M):
    print(f"  {month_of(m.start())}: {m.group(0).strip()}")

lifi_changelog_and_vaultpilot_issue_799_raw.jsonl: read snapshot, 3 calls, fetched 2026-10-28T22:08:38+00:00 to 2026-10-28T22:08:39+00:00 (no network call)

issue #799 "security: INTERMEDIATE_CHAIN_BRIDGES relaxes the chainId gate on a FALSE premise — 1885080…" by angelazhy, created 2026-07-19T21:28:00Z, state closed, closed 2026-09-17T16:32:27Z; https://github.com/agenthill/vaultpilot-mcp/issues/799
issue body lines that name a chain id, /v1/chains or a chain count:
  | The module pins `{ bridgeName: "near", intermediateChainId: 1885080386571452n }`, documented as NEAR Intents' intermediate settlement chain.
  | **`1885080386571452` is TRON's chain id in LiFi's on-chain `BridgeData` namespace** — distinct from the API-namespace `728126428` the codebase pins in `LIFI_CHAIN_ID` (`swap/index.ts:38-46`, duplicated at `tron/lifi-swap
  | | ETH→TRON USDC, **unforced** | `symbiosis` | `"symbiosis"` | **1885080386571452** | 728126428 |
  | **The second row is the falsifier and it fires.** If 

## 3. Tokens

`GET /v1/tokens?chains=<id>&chainTypes=…`, one call per chain id, for the six source chains and every destination id in the data (the endpoint rejects an id it does not know with HTTP 400, so ids are not batched). Then `GET /v1/token?chain=<id>&token=<address>` for every `sendingAssetId` the bulk list misses. Snapshot `lifi_api_tokens_by_chain.csv`: chain id, address (as the API returns it; EVM addresses are checksummed, so matching lowercases them), symbol, name, decimals, coinKey, LI.FI's `priceUSD` at fetch time, verification status, tags (empty on every row: the bulk list carries no `tags` key and the five single lookups returned none), and `fetched_via` (`tokens` for the bulk list, `token` for a single lookup). The per-chain outcome of the bulk fetch is kept in the manifest note.

In [9]:
WANTED_CHAIN_IDS = sorted(set(SOURCE_CHAIN_ID.values()) | {int(i) for i in tf["destinationChainId"].unique()})
TOKEN_COLS = ["chainId", "address", "symbol", "name", "decimals", "coinKey", "priceUSD", "verificationStatus", "tags", "fetched_via"]
TOKENS_URL = f"{LIFI}/tokens?chains=<id>&chainTypes={CHAIN_TYPES} and {LIFI}/token?chain=<id>&token=<address>"
FETCH_LOG = {}


def token_row(t, via):
    return {"chainId": t.get("chainId"), "address": t.get("address"), "symbol": t.get("symbol"), "name": t.get("name"), "decimals": t.get("decimals"),
            "coinKey": t.get("coinKey"), "priceUSD": t.get("priceUSD"), "verificationStatus": t.get("verificationStatus"),
            "tags": "|".join(t.get("tags") or []), "fetched_via": via}


def fetch_tokens():
    rows = []
    for cid in WANTED_CHAIN_IDS:
        status, body = get_json(f"{LIFI}/tokens", params={"chains": str(cid), "chainTypes": CHAIN_TYPES}, pause=0.2)
        toks = (body.get("tokens") or {}).get(str(cid), []) if status == 200 and isinstance(body, dict) else []
        FETCH_LOG[cid] = {"http_status": status, "tokens": len(toks)}
        rows += [token_row(t, "tokens") for t in toks]
    bulk = pd.DataFrame(rows, columns=TOKEN_COLS)
    known = set(zip(bulk["chainId"].astype(int), bulk["address"].str.lower()))
    pairs = tf[["chain", "sendingAssetId"]].drop_duplicates()
    misses = [(SOURCE_CHAIN_ID[c], a) for c, a in zip(pairs["chain"], pairs["sendingAssetId"]) if (SOURCE_CHAIN_ID[c], a) not in known]
    print(f"bulk lists: {len(bulk)} tokens over {sum(v['tokens'] > 0 for v in FETCH_LOG.values())} chains; sendingAssetIds not in them: {len(misses)}, looked up one by one")
    single = []
    for cid, addr in misses:
        status, body = get_json(f"{LIFI}/token", params={"chain": str(cid), "token": addr}, pause=0.2)
        FETCH_LOG[f"token:{cid}:{addr}"] = {"http_status": status, "tokens": int(status == 200)}
        if status == 200 and isinstance(body, dict):
            single.append(token_row(body, "token"))
    return pd.concat([bulk, pd.DataFrame(single, columns=TOKEN_COLS)], ignore_index=True)


def tokens_note():
    rejected = [str(k) for k, v in FETCH_LOG.items() if isinstance(k, int) and v["http_status"] != 200]
    empty = [str(k) for k, v in FETCH_LOG.items() if isinstance(k, int) and v["http_status"] == 200 and v["tokens"] == 0]
    singles = [k for k in FETCH_LOG if isinstance(k, str)]
    singles_ok = [k for k in singles if FETCH_LOG[k]["http_status"] == 200]
    return (f"one /v1/tokens call per chain id for {len(WANTED_CHAIN_IDS)} ids (6 source + destination ids); HTTP 400 for ids {rejected}; "
            f"HTTP 200 with an empty list for ids {empty}; {len(singles)} single /v1/token lookups for sendingAssetIds missing from the bulk lists, {len(singles_ok)} found")


tokens = snapshot_csv("lifi_api_tokens_by_chain.csv", fetch_tokens, "01 §3", TOKENS_URL, note=tokens_note,
                      read_kwargs={"dtype": {"address": str, "symbol": str, "name": str, "coinKey": str, "priceUSD": str, "verificationStatus": str, "tags": str, "fetched_via": str}})
tokens["address_lc"] = tokens["address"].str.lower()
print("lifi_api_tokens_by_chain.csv rows:", len(tokens), " chains with tokens:", tokens["chainId"].nunique(), " fetched_via:", tokens["fetched_via"].value_counts().to_dict())
print("destination ids with no token list:", sorted(set(WANTED_CHAIN_IDS) - set(tokens["chainId"].unique())))
tokens.groupby("chainId").size().sort_values(ascending=False).head(12).to_frame("tokens").T

lifi_api_tokens_by_chain.csv: read snapshot, 35903 rows, fetched 2026-10-28T14:48:32+00:00 (no network call)
lifi_api_tokens_by_chain.csv rows: 35903  chains with tokens: 63  fetched_via: {'tokens': 35898, 'token': 5}
destination ids with no token list: [204, 232, 60808, 81457, 1201081091099710, 1885080386571452]


chainId,1,1151111081099710,999,56,42161,10,143,57073,5000,137,728126428,8453
tokens,6654,5958,3734,2958,2738,1931,1673,1530,1519,1441,1207,1172


## 4. Tools

`GET /v1/tools` returns `bridges` and `exchanges`, each with `key`, `name` and the chains it supports. Snapshot `lifi_api_tools_bridges_and_exchanges.csv`: kind (`bridge` or `exchange`), key, name, number of supported chains (or chain pairs).

In [10]:
def fetch_tools():
    status, body = get_json(f"{LIFI}/tools")
    assert status == 200, (status, body)
    rows = [{"kind": kind[:-1], "key": t["key"], "name": t["name"], "n_supported_chains": len(t.get("supportedChains") or [])}
            for kind in ("bridges", "exchanges") for t in body[kind]]
    return pd.DataFrame(rows)


tools = snapshot_csv("lifi_api_tools_bridges_and_exchanges.csv", fetch_tools, "01 §4", f"{LIFI}/tools", read_kwargs={"dtype": {"key": str, "name": str, "kind": str}})
print(tools["kind"].value_counts().to_dict())
tools[tools["kind"] == "bridge"].sort_values("key")[["key", "name", "n_supported_chains"]].reset_index(drop=True).T

lifi_api_tools_bridges_and_exchanges.csv: read snapshot, 79 rows, fetched 2026-10-28T14:48:56+00:00 (no network call)
{'exchange': 41, 'bridge': 38}


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37
key,across,allbridge,arbitrum,cctp,celercircle,celercirclefast,centrifuge,chainflip,eco,frax,garden,gasZipBridge,glacis,gnosis,hyperliquidNative,hyperliquidSA,layerswap,lifiIntents,m0,mayan,mayanFastMCTP,mayanMCTP,mayanWH,megaeth,near,omni,paxos,polygon,polymer,polymerStandard,relaydepository,smartDeposits,squid,stargateV2,stargateV2Bus,superset,symbiosis,unit
name,AcrossV4,Allbridge,Arbitrum Bridge,Circle CCTP,CCTP + Celer (Standard),CCTP + Celer (Fast),Centrifuge,Chainflip,Eco,Frax Bridge,Garden,GasZip,Glacis,Gnosis Bridge,Hyperliquid Native,Hyperliquid,Layerswap,LI.FI Intents,M0,Mayan (Swift),CCTPv2 + Mayan,CCTP + Mayan,Mayan (Wormhole),MegaETH Gateway,NearIntents,Omni Bridge,Paxos Labs Transit,Polygon Bridge (PoS),Polymer (Fast),Polymer (Standard),Relay,Smart Deposits,Squid,StargateV2 (Fast mode),StargateV2 (Economy mode),Superset,Symbiosis,Unit
n_supported_chains,332,156,1,11,42,42,2,10,225,441,22,3132,780,1,2,2,1849,714,6,128,164,164,2,1,238,1,6,1,439,439,1681,202,156,830,830,6,846,12


## 5. Status sample

`GET /v1/status?txHash=<tx_hash>` for a random sample of events: 100 rows per source chain (600 in all), drawn with `DataFrameGroupBy.sample(n=100, random_state=20261028)` from the parsed transfers in file order, 0.2 s between calls, backoff on HTTP 429. The sample is by event row; for the 11 transactions that emit two events the endpoint is keyed by transaction hash, so `api_transactionId` may differ from the event's `transactionId` (checked in §10).

Snapshot `lifi_api_transfer_status_sample_600_raw.jsonl`: one line per sampled event with the event keys, HTTP status and the raw response, appended as each call returns (a re-run resumes from it). Snapshot `lifi_api_transfer_status_sample_600.csv`: the sample flattened to the event fields plus `status`, `substatus`, `tool`, sending and receiving chain, token, amount, USD, gas USD, and the quote's `fromAmountUSD`, `toAmountUSD`, fee costs (total and the part whose name contains "LIFI") and gas costs.

In [11]:
SEED, PER_CHAIN = 20261028, 100
STATUS_URL = f"{LIFI}/status?txHash=<tx_hash>"
JSONL = REF / "lifi_api_transfer_status_sample_600_raw.jsonl"
sample = (tf.groupby("chain", group_keys=False).sample(n=PER_CHAIN, random_state=SEED)
            .sort_values(["chain", "block_time", "log_index"], kind="stable").reset_index(drop=True))
print("sampled rows:", len(sample), sample["chain"].value_counts().to_dict())


def flatten_status(ev, rec):
    body = rec["response"]
    out = {"chain": ev.chain, "tx_hash": ev.tx_hash, "log_index": int(ev.log_index), "block_time": ev.block_time.isoformat(),
           "event_transactionId": ev.transactionId, "event_bridge": ev.bridge, "event_integrator": ev.integrator, "event_sendingAssetId": ev.sendingAssetId,
           "event_minAmount": str(ev.minAmount), "event_destinationChainId": int(ev.destinationChainId), "event_hasSourceSwaps": bool(ev.hasSourceSwaps),
           "event_hasDestinationCall": bool(ev.hasDestinationCall), "amount_usd_provided": ev.amount_usd_provided,
           "http_status": rec["http_status"], "fetched_at_utc": rec["fetched_at_utc"]}
    if not isinstance(body, dict):
        out["api_message"] = str(body)[:200]
        return out
    md_ = body.get("metadata") or {}
    out.update({"api_transactionId": body.get("transactionId"), "api_message": body.get("message"), "status": body.get("status"), "substatus": body.get("substatus"),
                "substatusMessage": body.get("substatusMessage"), "tool": body.get("tool"), "fromAddress": body.get("fromAddress"), "toAddress": body.get("toAddress"),
                "metadata_integrator": md_.get("integrator")})
    for side in ("sending", "receiving"):
        s = body.get(side) or {}
        tok = s.get("token") or {}
        out.update({f"{side}_txHash": s.get("txHash"), f"{side}_chainId": s.get("chainId"), f"{side}_token_address": tok.get("address"),
                    f"{side}_token_symbol": tok.get("symbol"), f"{side}_token_decimals": tok.get("decimals"), f"{side}_amount": s.get("amount"),
                    f"{side}_amountUSD": s.get("amountUSD"), f"{side}_gasAmountUSD": s.get("gasAmountUSD"), f"{side}_timestamp": s.get("timestamp"),
                    f"{side}_included_tools": "|".join(str(st.get("tool")) for st in (s.get("includedSteps") or []))})
    q = body.get("quote") or {}
    est = q.get("estimate") or {}
    fees, gas = est.get("feeCosts") or [], est.get("gasCosts") or []
    usd = lambda items: sum(float(i.get("amountUSD") or 0) for i in items)
    out.update({"quote_tool": q.get("tool"), "quote_integrator": q.get("integrator"), "quote_fromAmountUSD": est.get("fromAmountUSD"), "quote_toAmountUSD": est.get("toAmountUSD"),
                "quote_fee_usd": usd(fees) if fees else None,
                "quote_fee_lifi_usd": usd([f for f in fees if "lifi" in str(f.get("name", "")).lower().replace(".", "")]) if fees else None,
                "quote_fee_names": "|".join(str(f.get("name")) for f in fees), "quote_gas_usd": usd(gas) if gas else None})
    return out


def fetch_status_sample():
    done = {}
    if JSONL.exists():
        for line in JSONL.read_text().splitlines():
            rec = json.loads(line)
            done[(rec["chain"], rec["tx_hash"], rec["log_index"])] = rec
        print(f"resuming: {len(done)} responses already in {JSONL.name}")
    with JSONL.open("a") as fh:
        for i, ev in enumerate(sample.itertuples(index=False)):
            key = (ev.chain, ev.tx_hash, int(ev.log_index))
            if key in done:
                continue
            status, body = get_json(f"{LIFI}/status", params={"txHash": ev.tx_hash}, pause=0.2)
            rec = {"chain": ev.chain, "tx_hash": ev.tx_hash, "log_index": int(ev.log_index), "event_transactionId": ev.transactionId,
                   "fetched_at_utc": now_utc(), "http_status": status, "response": body}
            fh.write(json.dumps(rec) + "\n")
            fh.flush()
            done[key] = rec
            if (i + 1) % 100 == 0:
                print(f"  {i + 1}/{len(sample)} done")
    return pd.DataFrame([flatten_status(ev, done[(ev.chain, ev.tx_hash, int(ev.log_index))]) for ev in sample.itertuples(index=False)])


STR_COLS = ["tx_hash", "event_transactionId", "event_sendingAssetId", "event_minAmount", "api_transactionId", "sending_txHash", "receiving_txHash",
            "sending_token_address", "receiving_token_address", "sending_amount", "receiving_amount", "fromAddress", "toAddress", "status", "substatus",
            "tool", "quote_tool", "quote_fee_names", "sending_included_tools", "receiving_included_tools"]
status_sample = snapshot_csv("lifi_api_transfer_status_sample_600.csv", fetch_status_sample, "01 §5", STATUS_URL,
                             note=f"{PER_CHAIN} events per source chain, DataFrameGroupBy.sample(random_state={SEED}); raw responses in lifi_api_transfer_status_sample_600_raw.jsonl",
                             read_kwargs={"dtype": {c: str for c in STR_COLS}})
if not JSONL.exists():
    raise FileNotFoundError(f"{JSONL} is missing: the raw responses must be committed next to lifi_api_transfer_status_sample_600.csv")
manifest = manifest_read()
if "lifi_api_transfer_status_sample_600_raw.jsonl" not in manifest:
    manifest_write("lifi_api_transfer_status_sample_600_raw.jsonl", STATUS_URL, sum(1 for _ in JSONL.open()), "01 §5", manifest["lifi_api_transfer_status_sample_600.csv"]["fetched_at_utc"],
                   note="raw JSON responses, one line per sampled event, with the event keys and the HTTP status")
print("lifi_api_transfer_status_sample_600.csv:", status_sample.shape, " HTTP statuses:", status_sample["http_status"].value_counts().to_dict())

sampled rows: 600 {'arbitrum': 100, 'base': 100, 'bnb': 100, 'ethereum': 100, 'optimism': 100, 'polygon': 100}
lifi_api_transfer_status_sample_600.csv: read snapshot, 600 rows, fetched 2026-10-28T16:00:01+00:00 (no network call)
lifi_api_transfer_status_sample_600.csv: (600, 52)  HTTP statuses: {200: 600}


## 6. CoinGecko hourly prices

`GET /coins/<platform>/contract/<address>/market_chart/range?vs_currency=usd&from=<unix>&to=<unix>` for ERC-20 tokens (platform ids: `ethereum`, `arbitrum-one`, `base`, `optimistic-ethereum`, `polygon-pos`, `binance-smart-chain`) and `GET /coins/<coin_id>/market_chart/range` for native assets (`ethereum` for the zero address on ethereum, arbitrum, base and optimism; `binancecoin` on bnb; on polygon `matic-network` is tried first and `polygon-ecosystem-token` second, the lookup log records which returned points). Window 2026-10-13 00:00 to 2026-10-20 00:00 UTC; a range under 90 days returns hourly points, which are floored to the hour and averaged when more than one falls in an hour.

Without a key (`COINGECKO_API_KEY` in the environment, else in the gitignored `.env` at the repository root, parsed as plain `KEY=VALUE` lines) the public endpoint allows roughly five calls a minute (HTTP 429 with `Retry-After: 60` beyond that, measured during development), so calls are paced at 8 s and slow down further on each 429; with the key the demo header is sent and calls are paced at 2.5 s. One request per distinct endpoint (the four ETH natives share one). Progress is written after every pair, so an interrupted run resumes: `coingecko_request_log.csv` (one row per request: chain, token, URL, HTTP status, points, hours in the window) and `coingecko_hourly_prices_2026-10-13_to_2026-10-19.partial.csv`, which becomes `coingecko_hourly_prices_2026-10-13_to_2026-10-19.csv` (chain, token_address, hour, price_usd, n_points) when every pair has been tried.

In [12]:
CG_PLATFORM = {"ethereum": "ethereum", "arbitrum": "arbitrum-one", "base": "base", "optimism": "optimistic-ethereum", "polygon": "polygon-pos", "bnb": "binance-smart-chain"}
CG_NATIVE = {"ethereum": ["ethereum"], "arbitrum": ["ethereum"], "base": ["ethereum"], "optimism": ["ethereum"], "polygon": ["matic-network", "polygon-ecosystem-token"], "bnb": ["binancecoin"]}
CG_FROM, CG_TO = int(EXPORT_START.timestamp()), int(EXPORT_END.timestamp())
CG_HEADERS = {"x-cg-demo-api-key": COINGECKO_KEY} if COINGECKO_KEY else {}
CG_PAUSE = 2.5 if COINGECKO_KEY else 8.0
CG_URL = f"{COINGECKO}/coins/<platform>/contract/<address>/market_chart/range?vs_currency=usd&from={CG_FROM}&to={CG_TO} and {COINGECKO}/coins/<coin_id>/market_chart/range?…"
LOOKUPS, PARTIAL = REF / "coingecko_request_log.csv", REF / "coingecko_hourly_prices_2026-10-13_to_2026-10-19.partial.csv"
CG_PAIRS = (tf[["chain", "sendingAssetId"]].drop_duplicates().rename(columns={"sendingAssetId": "token_address"})
              .sort_values(["chain", "token_address"]).reset_index(drop=True))
print("(chain, token) pairs to price:", len(CG_PAIRS))


def cg_candidates(chain, token_address):
    if token_address == ZERO_ADDRESS:
        return [f"{COINGECKO}/coins/{cid}/market_chart/range" for cid in CG_NATIVE[chain]]
    return [f"{COINGECKO}/coins/{CG_PLATFORM[chain]}/contract/{token_address}/market_chart/range"]


def hourly(points):
    df = pd.DataFrame(points, columns=["ts_ms", "price_usd"])
    df["hour"] = pd.to_datetime(df["ts_ms"], unit="ms", utc=True).dt.floor("h")
    h = df.groupby("hour")["price_usd"].agg(price_usd="mean", n_points="size").reset_index()
    return h[(h["hour"] >= EXPORT_START) & (h["hour"] < EXPORT_END)]


def fetch_coingecko():
    done = set()
    if LOOKUPS.exists():
        lk = pd.read_csv(LOOKUPS, dtype=str)
        done = set(zip(lk["chain"], lk["token_address"]))
        print(f"resuming: {len(done)} pairs already in {LOOKUPS.name}")
    cache, pause, seen_429 = {}, CG_PAUSE, THROTTLE["count"]
    todo = [(c, a) for c, a in zip(CG_PAIRS["chain"], CG_PAIRS["token_address"]) if (c, a) not in done]
    t_start = time.time()
    for i, (chain, addr) in enumerate(todo):
        for url in cg_candidates(chain, addr):
            if url not in cache:
                cache[url] = get_json(url, params={"vs_currency": "usd", "from": CG_FROM, "to": CG_TO}, headers=CG_HEADERS, pause=pause)
                if THROTTLE["count"] > seen_429:
                    seen_429, pause = THROTTLE["count"], min(pause + 2.0, 20.0)
            status, body = cache[url]
            pts = body.get("prices") or [] if status == 200 and isinstance(body, dict) else []
            h = hourly(pts) if pts else pd.DataFrame(columns=["hour", "price_usd", "n_points"])
            append_csv(LOOKUPS, pd.DataFrame([{"chain": chain, "token_address": addr, "request_url": url, "http_status": status, "n_points": len(pts),
                                                "hours_in_window": len(h), "fetched_at_utc": now_utc()}]))
            if len(h):
                append_csv(PARTIAL, h.assign(chain=chain, token_address=addr)[["chain", "token_address", "hour", "price_usd", "n_points"]])
                break
        if (i + 1) % 25 == 0:
            print(f"  {i + 1}/{len(todo)} pairs, {(time.time() - t_start) / 60:.1f} min, pause {pause:.0f}s, 429s so far {THROTTLE['count']}")
    out = pd.read_csv(PARTIAL, dtype={"token_address": str}) if PARTIAL.exists() else pd.DataFrame(columns=["chain", "token_address", "hour", "price_usd", "n_points"])
    out = out.sort_values(["chain", "token_address", "hour"]).reset_index(drop=True)
    if PARTIAL.exists():
        PARTIAL.unlink()
    return out


def coingecko_note():
    lk = pd.read_csv(LOOKUPS, dtype={"token_address": str})
    priced = lk[lk["hours_in_window"].astype(int) > 0].drop_duplicates(["chain", "token_address"])
    return (f"{len(lk)} requests for {len(CG_PAIRS)} (chain, token) pairs; pairs with at least one hour: {len(priced)}; "
            f"HTTP statuses {lk['http_status'].astype(int).value_counts().to_dict()}; key used: {bool(COINGECKO_KEY)}; request log in coingecko_request_log.csv")


prices_cg = snapshot_csv("coingecko_hourly_prices_2026-10-13_to_2026-10-19.csv", fetch_coingecko, "01 §6", CG_URL, note=coingecko_note, read_kwargs={"dtype": {"token_address": str}})
prices_cg["hour"] = pd.to_datetime(prices_cg["hour"], utc=True)
lookups = pd.read_csv(LOOKUPS, dtype={"token_address": str})
manifest = manifest_read()
if "coingecko_request_log.csv" not in manifest:
    manifest_write("coingecko_request_log.csv", CG_URL, len(lookups), "01 §6", manifest["coingecko_hourly_prices_2026-10-13_to_2026-10-19.csv"]["fetched_at_utc"],
                   note="one row per CoinGecko request: chain, token, request URL (no key), HTTP status, points returned, hours inside the window")
print("coingecko_hourly_prices_2026-10-13_to_2026-10-19.csv:", prices_cg.shape, " pairs with a series:", prices_cg[["chain", "token_address"]].drop_duplicates().shape[0], "of", len(CG_PAIRS))
print("hours per pair:", prices_cg.groupby(["chain", "token_address"]).size().value_counts().sort_index().to_dict())
print("points per hour:", prices_cg["n_points"].value_counts().sort_index().to_dict())
print("requests by HTTP status:", lookups["http_status"].value_counts().to_dict(), " 429s seen during this run:", THROTTLE["count"])
lookups[lookups["request_url"].str.contains("matic-network|polygon-ecosystem-token|binancecoin|coins/ethereum/")][["chain", "token_address", "request_url", "http_status", "n_points", "hours_in_window"]]

(chain, token) pairs to price: 255


coingecko_hourly_prices_2026-10-13_to_2026-10-19.csv: read snapshot, 41777 rows, fetched 2026-10-28T16:06:54+00:00 (no network call)


coingecko_hourly_prices_2026-10-13_to_2026-10-19.csv: (41777, 5)  pairs with a series: 249 of 255


hours per pair: {129: 1, 152: 1, 168: 247}
points per hour: {1: 41777}
requests by HTTP status: {200: 250, 404: 6}  429s seen during this run: 0


,chain,token_address,request_url,http_status,n_points,hours_in_window
0,arbitrum,0x0000000000000000000000000000000000000000,https://api.coingecko.com/api/v3/coins/ethereum/market_c...,200,169,168
35,base,0x0000000000000000000000000000000000000000,https://api.coingecko.com/api/v3/coins/ethereum/market_c...,200,169,168
90,bnb,0x0000000000000000000000000000000000000000,https://api.coingecko.com/api/v3/coins/binancecoin/marke...,200,169,168
116,ethereum,0x0000000000000000000000000000000000000000,https://api.coingecko.com/api/v3/coins/ethereum/market_c...,200,169,168
117,ethereum,0x00000000efe302beaa2b3e6e1b18d08d69a9012a,https://api.coingecko.com/api/v3/coins/ethereum/contract...,200,169,168
118,ethereum,0x0f5d2fb29fb7d3cfee444a200298f468908cc942,https://api.coingecko.com/api/v3/coins/ethereum/contract...,200,169,168
119,ethereum,0x0f81001ef0a83ecce5ccebf63eb302c70a39a654,https://api.coingecko.com/api/v3/coins/ethereum/contract...,200,169,168
120,ethereum,0x122940c4c5f9ccfae7fa86455a42d3ec140855ce,https://api.coingecko.com/api/v3/coins/ethereum/contract...,200,153,152
121,ethereum,0x1a7e4e63778b4f12a199c062f3efdd288afcbce8,https://api.coingecko.com/api/v3/coins/ethereum/contract...,200,169,168
122,ethereum,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,https://api.coingecko.com/api/v3/coins/ethereum/contract...,200,169,168


In [13]:
# Pairs CoinGecko did not price, with the LI.FI token identity and the rows they carry (reference data).
cg_missing = CG_PAIRS.merge(prices_cg[["chain", "token_address"]].drop_duplicates(), how="left", indicator=True).query("_merge == 'left_only'").drop(columns="_merge")
cg_missing = cg_missing.merge(lookups.groupby(["chain", "token_address"])["http_status"].agg(lambda s: "|".join(map(str, s))).reset_index(), how="left")
cg_missing = cg_missing.merge(tf.groupby(["chain", "sendingAssetId"]).agg(rows=("tx_hash", "size"), usd_provided=("amount_usd_provided", "sum")).reset_index()
                              .rename(columns={"sendingAssetId": "token_address"}), how="left")
cg_missing["chainId"] = cg_missing["chain"].map(SOURCE_CHAIN_ID)
cg_missing = cg_missing.merge(tokens[["chainId", "address_lc", "symbol", "name", "decimals"]].drop_duplicates(["chainId", "address_lc"]),
                              left_on=["chainId", "token_address"], right_on=["chainId", "address_lc"], how="left").drop(columns="address_lc")
cg_missing = cg_missing.merge(prices[["chain", "token_address", "symbol"]].drop_duplicates().rename(columns={"symbol": "symbol_prices"}), how="left")
print("pairs without a CoinGecko series:", len(cg_missing), " rows they carry:", int(cg_missing["rows"].sum()), f" USD (provided): {cg_missing['usd_provided'].sum():,.2f}")
cg_missing.sort_values("rows", ascending=False)

pairs without a CoinGecko series: 6  rows they carry: 11  USD (provided): 406,586.76


,chain,token_address,http_status,rows,usd_provided,chainId,symbol,name,decimals,symbol_prices
2,ethereum,0xb0f70c0bd6fd87dbeb7c10dc692a2a6106817072,404,3,406390.318019,1,BTC.b,Bitcoin,8,BTC.b
4,optimism,0x43edd7f3831b08fe70b7555ddd373c8bf65a9050,404,3,0.000000,10,frxETH,Frax Ether,18,NaN
3,ethereum,0xb197e02499e6502733c6bce2eb39013c39a03147,404,2,37.697431,1,PROS,Wrapped PROS,18,PROS
0,arbitrum,0x03569cc076654f82679c4ba2124d64774781b01d,404,1,0.000000,42161,BOLD,BOLD Stablecoin,18,NaN
1,base,0x9cd44eca3d1e0ada6fbdaf5793031c3a500089b1,404,1,157.292308,8453,SPARKLET,UPLAND,18,SPARKLET
5,optimism,0x9485aca5bbbe1667ad97c7fe7c4531a624c8b1ed,404,1,1.457091,10,EURA,EURA (previously agEUR),18,EURA


## 7. Destination ids resolved

Every `destinationChainId` in the data is joined to the chain-id registry of §2.1 (reference data). Each id gets its name, key, chain type and LI.FI namespace, its rows and USD (provided feed only), its receiver-sentinel rows (`0x11f1…f1`, which is `NON_EVM_ADDRESS` in `LiFiData.sol`), and whether `/v1/chains` lists it, both in the §2 snapshot and with all six chain types.

The section also includes a table by chain type with Tron's two ids merged and shown apart, Tron under one chain, the timing and row features of the rows to the four EVM ids `/v1/chains` does not list, the size of the ids, and the ten destinations with the most rows.

In [14]:
dest = (tf.groupby("destinationChainId")
          .agg(rows=("tx_hash", "size"), usd_provided=("amount_usd_provided", "sum"), sentinel_rows=("receiver", lambda s: int((s == SENTINEL_RECEIVER).sum())))
          .reset_index())
dest = dest.merge(registry[["chain_id", "name", "key", "chain_type", "namespace", "api_chain_id", "name_source", "listed_in_lifi_api_chains_csv", "listed_by_v1_chains", "connections_http_status"]],
                  left_on="destinationChainId", right_on="chain_id", how="left").drop(columns="chain_id")
assert len(dest) == 69 and dest["name"].notna().all() and dest["chain_type"].notna().all()
dest["row_share"] = (dest["rows"] / len(tf)).round(4)
dest["usd_share"] = (dest["usd_provided"] / TOTAL_USD_PROVIDED).round(4)
dest["above_2_32"] = dest["destinationChainId"] > 2**32 - 1
dest = dest.sort_values("rows", ascending=False).reset_index(drop=True)
print("destination ids:", len(dest), " named by the registry:", int(dest["name"].notna().sum()), " listed by /v1/chains: in the §2 snapshot", int(dest["listed_in_lifi_api_chains_csv"].sum()),
      " with all six chain types", int(dest["listed_by_v1_chains"].sum()), " not listed:", int((~dest["listed_by_v1_chains"]).sum()))
print("names by source:", dest["name_source"].value_counts().to_dict())
pd.set_option("display.max_rows", 80)
dest[["destinationChainId", "name", "key", "chain_type", "namespace", "rows", "row_share", "usd_provided", "usd_share", "sentinel_rows", "listed_in_lifi_api_chains_csv", "listed_by_v1_chains"]].round(2)

destination ids: 69  named by the registry: 69  listed by /v1/chains: in the §2 snapshot 63  with all six chain types 64  not listed: 5
names by source: {'/v1/chains': 64, '@lifi/data-types supportedChains': 4, '/v1/chains name of 728126428, paired by LIFI_CHAIN_ID_TRON': 1}


,destinationChainId,name,key,chain_type,namespace,rows,row_share,usd_provided,usd_share,sentinel_rows,listed_in_lifi_api_chains_csv,listed_by_v1_chains
0,4663,Robinhood Chain,out,EVM,API and contract,27036,0.22,30183024.90,0.10,0,True,True
1,1,Ethereum,eth,EVM,API and contract,18893,0.15,75943939.18,0.25,0,True,True
2,8453,Base,bas,EVM,API and contract,14886,0.12,19915441.33,0.06,0,True,True
3,1151111081099710,Solana,sol,SVM,API and contract,14229,0.11,17001618.57,0.06,14229,True,True
4,56,BSC,bsc,EVM,API and contract,13334,0.11,11712981.09,0.04,0,True,True
5,42161,Arbitrum,arb,EVM,API and contract,11522,0.09,71069356.63,0.23,0,True,True
6,137,Polygon,pol,EVM,API and contract,5945,0.05,7038433.59,0.02,0,True,True
7,10,OP Mainnet,opt,EVM,API and contract,2888,0.02,5390550.61,0.02,0,True,True
8,999,HyperEVM,hyp,EVM,API and contract,2268,0.02,4163071.74,0.01,0,True,True
9,20000000000001,Bitcoin,btc,UTXO,API and contract,1568,0.01,4762485.05,0.02,1568,True,True


In [15]:
print("ids /v1/chains does not list with all six chain types, with the registry's name and its source:")
display(dest[~dest["listed_by_v1_chains"]][["destinationChainId", "name", "key", "chain_type", "namespace", "name_source", "api_chain_id", "rows", "usd_provided", "sentinel_rows", "connections_http_status"]].round(2))
print("ids the §2 snapshot did not list that /v1/chains lists with STL:", dest.loc[dest["listed_by_v1_chains"] & ~dest["listed_in_lifi_api_chains_csv"], ["destinationChainId", "name", "chain_type", "rows"]].to_dict("records"))

# By chain type. `chain` is the API id, so Tron's contract id 1885080386571452 and its API id 728126428 count as one chain; the table below it shows the two ids apart.
dest["chain"] = dest["api_chain_id"].astype("int64")


def type_table(d):
    t = d.groupby("chain_type").agg(chains=("chain", "nunique"), ids=("destinationChainId", "nunique"), rows=("rows", "sum"), usd_provided=("usd_provided", "sum"),
                                    sentinel_rows=("sentinel_rows", "sum"), rows_to_ids_not_in_v1_chains=("rows", lambda s: int(s[~d.loc[s.index, "listed_by_v1_chains"]].sum())))
    t = t.sort_values("rows", ascending=False)
    t.loc["total"] = [d["chain"].nunique(), d["destinationChainId"].nunique(), d["rows"].sum(), d["usd_provided"].sum(), d["sentinel_rows"].sum(), int(d.loc[~d["listed_by_v1_chains"], "rows"].sum())]
    t["row_share"] = (t["rows"] / len(tf)).round(4)
    t["usd_share"] = (t["usd_provided"] / TOTAL_USD_PROVIDED).round(4)
    return t


print("by chain type of the destination (Tron's two ids merged in TVM):")
display(type_table(dest).round({"usd_provided": 2, "row_share": 4, "usd_share": 4}))
print("TVM, Tron's two ids apart:")
display(dest[dest["chain"] == 728126428][["destinationChainId", "name", "namespace", "rows", "row_share", "usd_provided", "usd_share", "sentinel_rows", "listed_by_v1_chains"]].round(4))
print("chain names per type:", dest.groupby("chain_type")["name"].agg(lambda s: sorted(set(s))[:8]).to_dict())

print("sentinel-receiver rows by chain type of the destination:")
sent = tf[tf["receiver"] == SENTINEL_RECEIVER].merge(registry[["chain_id", "name", "chain_type"]], left_on="destinationChainId", right_on="chain_id", how="left")
display(sent.groupby(["chain_type", "destinationChainId", "name"]).size().to_frame("sentinel_rows"))
non_evm_ids = set(dest.loc[dest["chain_type"] != "EVM", "destinationChainId"])
print("rows to non-EVM destinations:", int(tf["destinationChainId"].isin(non_evm_ids).sum()), " of them with the sentinel:", int((tf["destinationChainId"].isin(non_evm_ids) & (tf["receiver"] == SENTINEL_RECEIVER)).sum()),
      " rows to EVM destinations with the sentinel:", int((~tf["destinationChainId"].isin(non_evm_ids) & (tf["receiver"] == SENTINEL_RECEIVER)).sum()))
print("rows to ids /v1/chains does not list, without the sentinel, by id and bridge:")
display(tf[tf["destinationChainId"].isin(set(dest.loc[~dest["listed_by_v1_chains"], "destinationChainId"])) & (tf["receiver"] != SENTINEL_RECEIVER)]
        .groupby(["destinationChainId", "bridge"]).size().to_frame("rows"))

ids /v1/chains does not list with all six chain types, with the registry's name and its source:


,destinationChainId,name,key,chain_type,namespace,name_source,api_chain_id,rows,usd_provided,sentinel_rows,connections_http_status
15,1885080386571452,Tron,trn,TVM,contract only,"/v1/chains name of 728126428, paired by LIFI_CHAIN_ID_TRON",728126428,685,9922643.17,685,400
55,81457,Blast,bls,EVM,API and contract,@lifi/data-types supportedChains,81457,12,666.73,0,400
56,204,opBNB,opb,EVM,API and contract,@lifi/data-types supportedChains,204,11,58.66,0,400
64,60808,BOB,boc,EVM,API and contract,@lifi/data-types supportedChains,60808,4,546.49,0,400
65,232,Lens,lns,EVM,API and contract,@lifi/data-types supportedChains,232,3,7.73,0,400


ids the §2 snapshot did not list that /v1/chains lists with STL: [{'destinationChainId': 1201081091099710, 'name': 'Stellar', 'chain_type': 'STL', 'rows': 4}]
by chain type of the destination (Tron's two ids merged in TVM):


,chains,ids,rows,usd_provided,sentinel_rows,rows_to_ids_not_in_v1_chains,row_share,usd_share
chain_type,,,,,,,,
EVM,62,62,108190,2.760571e+08,0,30,0.8665,0.8965
SVM,1,1,14229,1.700162e+07,14229,0,0.1140,0.0552
UTXO,2,2,1581,4.765927e+06,1581,0,0.0127,0.0155
TVM,1,2,687,9.922662e+06,687,685,0.0055,0.0322
MVM,1,1,167,1.874140e+05,167,0,0.0013,0.0006
STL,1,1,4,1.229500e+03,4,0,0.0000,0.0000
total,68,69,124858,3.079360e+08,16668,715,1.0000,1.0000


TVM, Tron's two ids apart:


,destinationChainId,name,namespace,rows,row_share,usd_provided,usd_share,sentinel_rows,listed_by_v1_chains
15,1885080386571452,Tron,contract only,685,0.0055,9.922643e+06,0.0322,685,False
67,728126428,Tron,API and contract,2,0.0000,1.893690e+01,0.0000,2,True


chain names per type: {'EVM': ['0G Mainnet', 'Abstract', 'Apechain', 'Arbitrum', 'Arbitrum Nova', 'Arc', 'Avalanche', 'BOB'], 'MVM': ['Sui'], 'STL': ['Stellar'], 'SVM': ['Solana'], 'TVM': ['Tron'], 'UTXO': ['Bitcoin', 'Zcash']}
sentinel-receiver rows by chain type of the destination:


sentinel_rows
chain_type destinationChainId name                  
MVM        9270000000000000   Sui                167
STL        1201081091099710   Stellar              4
SVM        1151111081099710   Solana           14229
TVM        728126428          Tron                 2
           1885080386571452   Tron               685
UTXO       20000000000001     Bitcoin           1568
           20000000000005     Zcash               13

rows to non-EVM destinations: 16668  of them with the sentinel: 16668  rows to EVM destinations with the sentinel: 0
rows to ids /v1/chains does not list, without the sentinel, by id and bridge:


rows
destinationChainId bridge               
204                gasZipBridge        4
                   layerswap           1
                   symbiosis           6
232                gasZipBridge        3
60808              gasZipBridge        1
                   layerswap           3
81457              layerswap           9
                   relaydepository     3

In [16]:
# Same-chain rows under the API's ids for the six source chains, and the chain-named bridges against the destination they reach (reference data).
tf["source_chain_id"] = tf["chain"].map(SOURCE_CHAIN_ID)
same = tf[tf["destinationChainId"] == tf["source_chain_id"]]
print("rows with destination == source chain id:", len(same), " by bridge:", same["bridge"].value_counts().to_dict())
chain_named = tf[tf["bridge"].isin(["arbitrum", "polygon", "gnosis", "omni", "megaeth", "superset"])]
display(chain_named.groupby(["bridge", "chain", "destinationChainId"]).size().to_frame("rows").reset_index()
        .merge(chains[["id", "name", "chainType"]], left_on="destinationChainId", right_on="id", how="left").drop(columns="id"))

rows with destination == source chain id: 295  by bridge: {'lifiIntentsDex': 284, 'm0Dex': 7, 'paxosDex': 4}


,bridge,chain,destinationChainId,rows,name,chainType
0,arbitrum,ethereum,42161,193,Arbitrum,EVM
1,gnosis,ethereum,100,6,Gnosis,EVM
2,megaeth,ethereum,4326,2,MegaETH,EVM
3,omni,ethereum,100,8,Gnosis,EVM
4,polygon,ethereum,137,14,Polygon,EVM
5,superset,arbitrum,8453,1,Base,EVM


In [17]:
# Tron under one chain: its contract id 1885080386571452 and its API id 728126428 (the pairing is reference data, §2.1; rows and USD are the data, provided feed).
tron = tf[tf["destinationChainId"].isin([728126428, 1885080386571452])]
display(tron.groupby(["destinationChainId", "bridge"]).agg(rows=("tx_hash", "size"), usd_provided=("amount_usd_provided", "sum"), sentinel_rows=("receiver", lambda s: int((s == SENTINEL_RECEIVER).sum())),
                                                          integrators=("integrator", "nunique"), first=("block_time", "min"), last=("block_time", "max")).round({"usd_provided": 2}))
print(f"Tron, both ids: {len(tron)} rows ({len(tron) / len(tf):.4%} of rows), USD (provided) {tron['amount_usd_provided'].sum():,.2f} ({tron['amount_usd_provided'].sum() / TOTAL_USD_PROVIDED:.4%}); "
      f"rows with the sentinel receiver {int((tron['receiver'] == SENTINEL_RECEIVER).sum())}")
print("the two rows under 728126428:")
display(tron[tron["destinationChainId"] == 728126428][["chain", "block_time", "bridge", "integrator", "tx_from", "receiver", "amount_usd_provided"]].round({"amount_usd_provided": 2}))

# The four EVM ids /v1/chains does not list: when their rows occur, and whether anything in the rows sets them apart from rows to the EVM ids it lists.
four_ids = sorted(dest.loc[~dest["listed_by_v1_chains"] & (dest["chain_type"] == "EVM"), "destinationChainId"])
four = tf[tf["destinationChainId"].isin(four_ids)]
last = four["block_time"].max()
print(f"\nthe four ids {four_ids} ({', '.join(registry.set_index('chain_id').loc[four_ids, 'name'])}): {len(four)} rows, first {four['block_time'].min()}, last {last}")
day = four["block_time"].dt.strftime("%Y-%m-%d")
print("rows by UTC day:", four.groupby(day).size().to_dict(), " by id and day:", four.groupby(["destinationChainId", day]).size().unstack(fill_value=0).to_dict("index"))
after = tf["block_time"] > last
print(f"rows in the data after {last}: {int(after.sum())} of {len(tf)} ({after.mean():.4f}); hours from it to the end of the window: {(EXPORT_END - last) / pd.Timedelta(hours=1):.1f}; "
      f"rows to the four ids among them: {int(four['block_time'].gt(last).sum())}")
fm = tf.loc[tf["bridge"] == "mayanFastMCTP", "block_time"].sort_values()
print(f"for comparison, mayanFastMCTP: {len(fm)} rows, the first at {fm.iloc[0]}, the second at {fm.iloc[1]}; rows at or before {last}: {int((fm <= last).sum())}; "
      f"rows per hour from its first row to the end of the window: {len(fm) / ((EXPORT_END - fm.iloc[0]) / pd.Timedelta(hours=1)):.2f}")
evm_listed = set(dest.loc[(dest["chain_type"] == "EVM") & dest["listed_by_v1_chains"], "destinationChainId"])
same_hours = tf[tf["destinationChainId"].isin(evm_listed) & (tf["block_time"] <= last)]


def features(d):
    return pd.Series({"rows": len(d), "receiver_20_byte_hex": d["receiver"].str.fullmatch(r"0x[0-9a-f]{40}").mean(), "receiver_equals_tx_from": (d["receiver"] == d["tx_from"]).mean(),
                      "receiver_is_sentinel": (d["receiver"] == SENTINEL_RECEIVER).mean(), "has_source_swaps": d["hasSourceSwaps"].mean(), "has_destination_call": d["hasDestinationCall"].mean(),
                      "referrer_non_zero": (d["referrer"] != ZERO_ADDRESS).mean(), "emitted_by_the_diamond": (d["contract_address"] == tf["contract_address"].iloc[0]).mean(),
                      "native_asset_sent": (d["sendingAssetId"] == ZERO_ADDRESS).mean(), "median_usd_provided": d["amount_usd_provided"].median()})


display(pd.DataFrame({"the four ids": features(four), "listed EVM ids, same hours": features(same_hours)}).round(3))
display(four.groupby(["destinationChainId", "bridge"]).agg(rows=("tx_hash", "size"), usd_provided=("amount_usd_provided", "sum"),
                                                          integrators=("integrator", lambda s: "|".join(f"{k}:{v}" for k, v in s.value_counts().items())),
                                                          source_chains=("chain", lambda s: "|".join(f"{k}:{v}" for k, v in s.value_counts().items()))).round(2))
print("rows with a non-zero referrer, by integrator:", four.loc[four["referrer"] != ZERO_ADDRESS, "integrator"].value_counts().to_dict(),
      " rows sending the native asset:", int((four["sendingAssetId"] == ZERO_ADDRESS).sum()), "of", len(four))
print("rows of the same bridges in the whole week:", {b: int((tf["bridge"] == b).sum()) for b in sorted(four["bridge"].unique())},
      " and of the same integrators:", {i: int((tf["integrator"] == i).sum()) for i in sorted(four["integrator"].unique())})

rows  usd_provided  sentinel_rows  integrators                     first                      last
destinationChainId bridge                                                                                                         
728126428          lifiIntents     2         18.94              2            1 2026-10-19 13:57:47+00:00 2026-10-19 14:00:18+00:00
1885080386571452   layerswap      76    1427616.82             76           12 2026-10-13 04:07:35+00:00 2026-10-19 21:07:23+00:00
                   near          513    6783442.59            513           22 2026-10-13 00:37:35+00:00 2026-10-19 22:59:47+00:00
                   symbiosis      96    1711583.76             96           12 2026-10-13 03:05:19+00:00 2026-10-19 18:12:20+00:00

Tron, both ids: 687 rows (0.5502% of rows), USD (provided) 9,922,662.11 (3.2223%); rows with the sentinel receiver 687
the two rows under 728126428:


,chain,block_time,bridge,integrator,tx_from,receiver,amount_usd_provided
116776,arbitrum,2026-10-19 13:57:47+00:00,lifiIntents,lifi-local-api,0x86dbd094bc7436bd106c53a6a137ab0ab810a6a9,0x11f111f111f111f111f111f111f111f111f111f1,9.97
116822,polygon,2026-10-19 14:00:18+00:00,lifiIntents,lifi-local-api,0x86dbd094bc7436bd106c53a6a137ab0ab810a6a9,0x11f111f111f111f111f111f111f111f111f111f1,8.97



the four ids [204, 232, 60808, 81457] (opBNB, Lens, BOB, Blast): 30 rows, first 2026-10-13 01:35:35+00:00, last 2026-10-14 12:55:11+00:00
rows by UTC day: {'2026-10-13': 15, '2026-10-14': 15}  by id and day: {204: {'2026-10-13': 8, '2026-10-14': 3}, 232: {'2026-10-13': 1, '2026-10-14': 2}, 60808: {'2026-10-13': 2, '2026-10-14': 2}, 81457: {'2026-10-13': 4, '2026-10-14': 8}}
rows in the data after 2026-10-14 12:55:11+00:00: 95904 of 124858 (0.7681); hours from it to the end of the window: 131.1; rows to the four ids among them: 0
for comparison, mayanFastMCTP: 771 rows, the first at 2026-10-14 12:47:39+00:00, the second at 2026-10-14 12:56:17+00:00; rows at or before 2026-10-14 12:55:11+00:00: 1; rows per hour from its first row to the end of the window: 5.88


,the four ids,"listed EVM ids, same hours"
rows,30.000,25105.000
receiver_20_byte_hex,1.000,1.000
receiver_equals_tx_from,1.000,0.793
receiver_is_sentinel,0.000,0.000
has_source_swaps,0.967,0.636
has_destination_call,0.000,0.047
referrer_non_zero,0.367,0.024
emitted_by_the_diamond,1.000,1.000
native_asset_sent,0.933,0.327
median_usd_provided,2.141,44.586


rows  usd_provided                                           integrators                       source_chains
destinationChainId bridge                                                                                                                       
204                gasZipBridge        4          4.06                                             safepal:4                               bnb:4
                   layerswap           1         34.73                                             safepal:1                          ethereum:1
                   symbiosis           6         19.87                                             safepal:6                               bnb:6
232                gasZipBridge        3          7.73                               hey:2|jumper.exchange:1                   ethereum:2|base:1
60808              gasZipBridge        1          0.25                                            lifi-api:1                          arbitrum:1
                   layerswap           3        546.24                                     jumper.exchange:3                   ethereum:2|base:1
81457              layerswap           9        151.72  jumper.exchange:5|rabbykmsv2:2|hot-wallet:1|tangem:1  base:5|ethereum:2|bnb:1|arbitrum:1
                   relaydepository     3        515.02                                     jumper.exchange:3                          ethereum:3

rows with a non-zero referrer, by integrator: {'safepal': 11}  rows sending the native asset: 28 of 30


rows of the same bridges in the whole week: {'gasZipBridge': 3678, 'layerswap': 12659, 'relaydepository': 31483, 'symbiosis': 772}  and of the same integrators: {'hey': 9, 'hot-wallet': 256, 'jumper.exchange': 40908, 'lifi-api': 3465, 'rabbykmsv2': 7759, 'safepal': 969, 'tangem': 2621}


In [18]:
# The size of the ids. Every one fits int64. Only Sui's exceeds 2^53 - 1, above which float64 does not hold every integer; the check is whether each id survives a float64 round trip.
ids = dest[["destinationChainId", "name", "chain_type"]].copy()
ids["above_2_32_minus_1"] = ids["destinationChainId"] > 2**32 - 1
ids["above_2_53_minus_1"] = ids["destinationChainId"] > 2**53 - 1
ids["float64_round_trip_exact"] = [int(float(int(i))) == int(i) for i in ids["destinationChainId"]]
print(f"largest id {ids['destinationChainId'].max():,} against int64's {2**63 - 1:,} and 2^53 - 1 = {2**53 - 1:,}")
print("ids above 2^32 - 1:", ids.loc[ids["above_2_32_minus_1"], ["destinationChainId", "name", "chain_type"]].to_dict("records"))
import math
print("ids above 2^53 - 1:", ids.loc[ids["above_2_53_minus_1"], ["destinationChainId", "name"]].to_dict("records"),
      f"; 9270000000000000 is even: {9270000000000000 % 2 == 0}; float64 spacing at that magnitude (math.ulp): {math.ulp(9270000000000000.0)}")
print("every id survives a float64 round trip:", bool(ids["float64_round_trip_exact"].all()))
print("non-EVM ids at or below 2^32 - 1 (the size rule misses them):", ids.loc[(ids["chain_type"] != "EVM") & ~ids["above_2_32_minus_1"], ["destinationChainId", "name", "chain_type"]].to_dict("records"),
      "; EVM ids above it:", ids.loc[(ids["chain_type"] == "EVM") & ids["above_2_32_minus_1"], "destinationChainId"].tolist())
print("LiFiData.sol's ids: the Stellar pattern in @lifi/types (ASCII codes of the key's letters + 1099710) checked on every ChainId member above 2^32:")
pat = chain_id_enum[chain_id_enum["chain_id"] > 2**32].assign(ascii_pattern=lambda d: [str(i) == "".join(str(ord(ch)) for ch in k) + "1099710" for i, k in zip(d["chain_id"], d["key"])])
display(pat[["member", "key", "chain_id", "section", "ascii_pattern"]])
print("LIFI_CHAIN_ID_TRON against the pattern for 'trn':", "".join(str(ord(ch)) for ch in "trn") + "1099710", "vs", int(lifidata.loc[lifidata["constant"] == "LIFI_CHAIN_ID_TRON", "chain_id"].iloc[0]))

# The ten destination ids with the most rows, for the profile's §6.
top10 = dest.head(10)[["destinationChainId", "name", "key", "chain_type", "rows", "usd_provided"]].copy()
top10["row_share_pct"] = (100 * top10["rows"] / len(tf)).round(1)
top10["usd_share_pct"] = (100 * top10["usd_provided"] / TOTAL_USD_PROVIDED).round(1)
top10["usd_provided_m"] = (top10["usd_provided"] / 1e6).round(2)
display(top10.drop(columns="usd_provided"))
print(f"top ten together: {top10['rows'].sum()} rows ({100 * top10['rows'].sum() / len(tf):.1f}%), USD {top10['usd_provided'].sum() / 1e6:.2f}M ({100 * top10['usd_provided'].sum() / TOTAL_USD_PROVIDED:.1f}%)")

largest id 9,270,000,000,000,000 against int64's 9,223,372,036,854,775,807 and 2^53 - 1 = 9,007,199,254,740,991
ids above 2^32 - 1: [{'destinationChainId': 1151111081099710, 'name': 'Solana', 'chain_type': 'SVM'}, {'destinationChainId': 20000000000001, 'name': 'Bitcoin', 'chain_type': 'UTXO'}, {'destinationChainId': 1885080386571452, 'name': 'Tron', 'chain_type': 'TVM'}, {'destinationChainId': 9270000000000000, 'name': 'Sui', 'chain_type': 'MVM'}, {'destinationChainId': 20000000000005, 'name': 'Zcash', 'chain_type': 'UTXO'}, {'destinationChainId': 1201081091099710, 'name': 'Stellar', 'chain_type': 'STL'}]
ids above 2^53 - 1: [{'destinationChainId': 9270000000000000, 'name': 'Sui'}] ; 9270000000000000 is even: True; float64 spacing at that magnitude (math.ulp): 2.0
every id survives a float64 round trip: True
non-EVM ids at or below 2^32 - 1 (the size rule misses them): [{'destinationChainId': 728126428, 'name': 'Tron', 'chain_type': 'TVM'}] ; EVM ids above it: []
LiFiData.sol's ids: th

,member,key,chain_id,section,ascii_pattern
76,TER,ter,1161011141099710,None-EVM (IDs are made up by the LI.FI team),True
77,OAS,oas,111971151099710,None-EVM (IDs are made up by the LI.FI team),True
78,SOL,sol,1151111081099710,SVM,True
79,FOG,fog,1021111031099710,SVM,True
80,SUI,sui,9270000000000000,MVM (IDs are made up by the LI.FI team),False
81,BTC,btc,20000000000001,UTXO (IDs are made up by the LI.FI team),False
82,BCH,bch,20000000000002,UTXO (IDs are made up by the LI.FI team),False
83,LTC,ltc,20000000000003,UTXO (IDs are made up by the LI.FI team),False
84,DGE,dge,20000000000004,UTXO (IDs are made up by the LI.FI team),False
85,ZEC,zec,20000000000005,UTXO (IDs are made up by the LI.FI team),False


LIFI_CHAIN_ID_TRON against the pattern for 'trn': 1161141101099710 vs 1885080386571452


,destinationChainId,name,key,chain_type,rows,row_share_pct,usd_share_pct,usd_provided_m
0,4663,Robinhood Chain,out,EVM,27036,21.7,9.8,30.18
1,1,Ethereum,eth,EVM,18893,15.1,24.7,75.94
2,8453,Base,bas,EVM,14886,11.9,6.5,19.92
3,1151111081099710,Solana,sol,SVM,14229,11.4,5.5,17.00
4,56,BSC,bsc,EVM,13334,10.7,3.8,11.71
5,42161,Arbitrum,arb,EVM,11522,9.2,23.1,71.07
6,137,Polygon,pol,EVM,5945,4.8,2.3,7.04
7,10,OP Mainnet,opt,EVM,2888,2.3,1.8,5.39
8,999,HyperEVM,hyp,EVM,2268,1.8,1.4,4.16
9,20000000000001,Bitcoin,btc,UTXO,1568,1.3,1.5,4.76


top ten together: 112569 rows (90.2%), USD 247.18M (80.3%)


## 8. Tokens resolved

Every (chain, `sendingAssetId`) pair in the data against `lifi_api_tokens_by_chain.csv` (reference data; addresses lowercased for the match) and against `prices.csv`: which tokens LI.FI does not know, whether decimals and symbol agree between the two sources, the names behind the duplicate `USDC` symbols the profile noticed, and the identity of the 11 pairs the provided feed does not price.

In [19]:
pairs = tf.groupby(["chain", "sendingAssetId"]).agg(rows=("tx_hash", "size"), usd_provided=("amount_usd_provided", "sum")).reset_index()
pairs["chainId"] = pairs["chain"].map(SOURCE_CHAIN_ID)
lifi_tok = tokens.drop_duplicates(["chainId", "address_lc"])[["chainId", "address_lc", "symbol", "name", "decimals", "coinKey", "verificationStatus", "tags", "fetched_via"]]
lifi_tok = lifi_tok.rename(columns={"symbol": "symbol_lifi", "name": "name_lifi", "decimals": "decimals_lifi"})
pm = prices[["chain", "token_address", "symbol", "decimals"]].drop_duplicates().rename(columns={"symbol": "symbol_prices", "decimals": "decimals_prices"})
res = (pairs.merge(lifi_tok, left_on=["chainId", "sendingAssetId"], right_on=["chainId", "address_lc"], how="left").drop(columns="address_lc")
            .merge(pm, left_on=["chain", "sendingAssetId"], right_on=["chain", "token_address"], how="left").drop(columns="token_address"))
res["known_to_lifi"] = res["symbol_lifi"].notna()
res["priced_by_feed"] = res["symbol_prices"].notna()
print("pairs:", len(res), " known to LI.FI:", int(res["known_to_lifi"].sum()), f"({res.loc[res['known_to_lifi'], 'rows'].sum()} rows)",
      " unknown:", int((~res["known_to_lifi"]).sum()), f"({res.loc[~res['known_to_lifi'], 'rows'].sum()} rows, USD provided {res.loc[~res['known_to_lifi'], 'usd_provided'].sum():,.2f})")
print("fetched_via among known pairs:", res.loc[res["known_to_lifi"], "fetched_via"].value_counts().to_dict())
print("verificationStatus among known pairs:", res.loc[res["known_to_lifi"], "verificationStatus"].value_counts(dropna=False).to_dict())
print("unknown to LI.FI:")
display(res[~res["known_to_lifi"]][["chain", "sendingAssetId", "rows", "usd_provided", "symbol_prices", "decimals_prices"]].round(2))

pairs: 255  known to LI.FI: 255 (124858 rows)  unknown: 0 (0 rows, USD provided 0.00)
fetched_via among known pairs: {'tokens': 250, 'token': 5}
verificationStatus among known pairs: {'verified': 254, 'unverified': 1}
unknown to LI.FI:


,chain,sendingAssetId,rows,usd_provided,symbol_prices,decimals_prices


In [20]:
both = res[res["known_to_lifi"] & res["priced_by_feed"]].copy()
both["decimals_agree"] = both["decimals_lifi"].astype(int) == both["decimals_prices"].astype(int)
both["symbol_agree"] = both["symbol_lifi"] == both["symbol_prices"]
both["symbol_agree_ci"] = both["symbol_lifi"].str.lower() == both["symbol_prices"].str.lower()
print("pairs in both sources:", len(both), " decimals agree:", int(both["decimals_agree"].sum()), " symbol agrees exactly:", int(both["symbol_agree"].sum()),
      " case-insensitively:", int(both["symbol_agree_ci"].sum()))
print("decimals disagreements:")
display(both[~both["decimals_agree"]][["chain", "sendingAssetId", "symbol_prices", "decimals_prices", "symbol_lifi", "decimals_lifi", "rows"]])
print("symbol disagreements (prices.csv symbol vs LI.FI symbol and name):")
display(both[~both["symbol_agree"]][["chain", "sendingAssetId", "symbol_prices", "symbol_lifi", "name_lifi", "decimals_prices", "rows", "usd_provided"]].round(2).sort_values("rows", ascending=False))

pairs in both sources: 244  decimals agree: 244  symbol agrees exactly: 235  case-insensitively: 236
decimals disagreements:


,chain,sendingAssetId,symbol_prices,decimals_prices,symbol_lifi,decimals_lifi,rows


symbol disagreements (prices.csv symbol vs LI.FI symbol and name):


,chain,sendingAssetId,symbol_prices,symbol_lifi,name_lifi,decimals_prices,rows,usd_provided
94,bnb,0x2170ed0880ac9a755fd29b2688956bd959f933f8,ETH,WETH,WETH,18.0,5433,1690911.87
248,polygon,0xc2132d05d31c914a87c6611c10748aeb04b58e8f,USDT0,USDT,USDT,6.0,2649,1056402.73
33,arbitrum,0xfd086bc7cd5c481dcc9c85ebe478a1c0b69fcbb9,USD₮0,USDT0,USD₮0,6.0,2173,3328402.55
217,optimism,0x01bff41798a0bcf287b996046ca68b395dbc1071,USD₮0,USDT0,USD₮0,6.0,111,75751.58
237,polygon,0x2791bca1f2de4661ed88a30c99a7a9449aa84174,USDC,USDC.e,Bridged USD Coin,6.0,72,77413.05
227,optimism,0x7f5c764cbc14f9669b88837ca1490cca17c31607,USDC,USDC.e,Bridged USD Coin,6.0,26,5740.18
34,arbitrum,0xff970a61a04b1ca14834a43f5de4533ebddb5cc8,USDC,USDC.e,Bridged USD Coin,6.0,13,1145.67
42,base,0x157a6df6b74f4e5e45af4e4615fde7b49225a662,ISLAND,ISLAND (niftyisland.com),ISLAND,18.0,1,0.00
174,ethereum,0x9f8f72aa9304c8b593d555f12ef6589cc3a579a2,MKR,mkr,Maker,18.0,1,83.65


In [21]:
# The USDC variants: every pair whose prices.csv symbol is USDC, or whose LI.FI symbol or name mentions USDC, by chain with LI.FI's name (reference data).
usdc = res[(res["symbol_prices"] == "USDC") | res["symbol_lifi"].fillna("").str.contains("USDC", case=False) | res["name_lifi"].fillna("").str.contains("USDC|USD Coin", case=False)]
display(usdc.sort_values(["chain", "rows"], ascending=[True, False])[["chain", "sendingAssetId", "symbol_prices", "decimals_prices", "symbol_lifi", "name_lifi", "decimals_lifi", "coinKey", "rows", "usd_provided"]].round(2))
print("chains where prices.csv has two USDC addresses:", res[res["symbol_prices"] == "USDC"].groupby("chain").size().loc[lambda s: s > 1].to_dict())
print("USDT, where the profile saw 18 decimals on bnb:")
display(res[res["symbol_prices"].isin(["USDT"])][["chain", "sendingAssetId", "symbol_lifi", "name_lifi", "decimals_prices", "decimals_lifi", "rows"]])

,chain,sendingAssetId,symbol_prices,decimals_prices,symbol_lifi,name_lifi,decimals_lifi,coinKey,rows,usd_provided
23,arbitrum,0xaf88d065e77c8cc2239327c5edb3a432268e5831,USDC,6.0,USDC,USD Coin,6,USDC,10638,6.017170e+07
28,arbitrum,0xeb466342c4d449bc9f53a865d5cb90586f405215,axlUSDC,6.0,axlUSDC,Axelar Wrapped USDC,6,axlUSDC,54,1.676480e+04
34,arbitrum,0xff970a61a04b1ca14834a43f5de4533ebddb5cc8,USDC,6.0,USDC.e,Bridged USD Coin,6,USDCe,13,1.145670e+03
61,base,0x833589fcd6edb6e08f4c7c32d4f71b54bda02913,USDC,6.0,USDC,USD Coin,6,USDC,17758,2.199970e+07
86,base,0xeb466342c4d449bc9f53a865d5cb90586f405215,axlUSDC,6.0,axlUSDC,Axelar Wrapped USDC,6,axlUSDC,116,5.193194e+04
106,bnb,0x8ac76a51cc950d9822d68b83fe1ad97b32cd580d,USDC,18.0,USDC,USD Coin,18,USDC,5308,5.395402e+06
97,bnb,0x4268b8f0b87b6eae5d897996e6b845ddbd99adf3,axlUSDC,6.0,axlUSDC,Axelar Wrapped USDC,6,axlUSDC,132,5.808070e+04
175,ethereum,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,6.0,USDC,USD Coin,6,USDC,8944,1.151893e+08
218,optimism,0x0b2c639c533813f4aa9d7837caf62653d097ff85,USDC,6.0,USDC,USD Coin,6,USDC,2123,3.377896e+06
227,optimism,0x7f5c764cbc14f9669b88837ca1490cca17c31607,USDC,6.0,USDC.e,Bridged USD Coin,6,USDCe,26,5.740180e+03


chains where prices.csv has two USDC addresses: {'arbitrum': 2, 'optimism': 2, 'polygon': 2}
USDT, where the profile saw 18 decimals on bnb:


,chain,sendingAssetId,symbol_lifi,name_lifi,decimals_prices,decimals_lifi,rows
89,base,0xfde4c96c8593536e31f229ea8f37b2ada2699bb2,USDT,Tether USD,6.0,6,999
99,bnb,0x55d398326f99059ff775485246999027b3197955,USDT,USDT,18.0,18,11884
204,ethereum,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,USDT,6.0,6,4453
230,optimism,0x94b008aa00579c1307b0ef2c499ad98a8ce58e58,USDT,USDT,6.0,6,347


In [22]:
# The 11 pairs the provided feed does not price: LI.FI identity, decimals and whether CoinGecko priced them (reference data).
unpriced = res[~res["priced_by_feed"]].merge(prices_cg.groupby(["chain", "token_address"]).size().to_frame("coingecko_hours").reset_index(),
                                            left_on=["chain", "sendingAssetId"], right_on=["chain", "token_address"], how="left").drop(columns="token_address")
unpriced["coingecko_hours"] = unpriced["coingecko_hours"].fillna(0).astype(int)
display(unpriced.sort_values("rows", ascending=False)[["chain", "sendingAssetId", "rows", "symbol_lifi", "name_lifi", "decimals_lifi", "verificationStatus", "coingecko_hours"]])
print("the polygon native asset in lifi_api_chains.csv:", chains.loc[chains["id"] == 137, ["name", "native_symbol", "native_name", "native_decimals"]].to_dict("records"))

,chain,sendingAssetId,rows,symbol_lifi,name_lifi,decimals_lifi,verificationStatus,coingecko_hours
9,polygon,0x0000000000000000000000000000000000000000,364,POL,Polygon Ecosystem Token,18,verified,168
1,arbitrum,0x80eede496655fb9047dd39d9f418d5483ed600df,3,frxUSD,Frax USD,18,verified,168
8,optimism,0x43edd7f3831b08fe70b7555ddd373c8bf65a9050,3,frxETH,Frax Ether,18,verified,0
4,bnb,0x02d608506ca0048d0d991a11f1e7fb8cad1e44f8,2,AALon,American Airlines Group,18,verified,168
10,polygon,0x80eede496655fb9047dd39d9f418d5483ed600df,2,frxUSD,Frax USD,18,verified,168
0,arbitrum,0x03569cc076654f82679c4ba2124d64774781b01d,1,BOLD,BOLD Stablecoin,18,verified,0
2,base,0x7094c27f342dbadfbbed005b219431595e33b305,1,QUICK,Quickswap,18,verified,168
3,base,0xd6122ddada244913521f3d62006eaf756c157660,1,BR,Bedrock,18,verified,168
5,bnb,0x91f9cc2649ac70a071602cade9b0c1a5868af51d,1,WXTZ,Wrapped XTZ,18,verified,168
6,bnb,0xec93fe7ff4b09ca3ccafbc4cc9615e62be412780,1,COPXon,Global X Copper Miners,18,verified,168


the polygon native asset in lifi_api_chains.csv: [{'name': 'Polygon', 'native_symbol': 'POL', 'native_name': 'Polygon Ecosystem Token', 'native_decimals': 18}]


## 9. Bridge strings against tools

The 31 distinct `bridge` strings in the data against `lifi_api_tools_bridges_and_exchanges.csv` (reference data): matched by exact key to a bridge, to an exchange, or to neither; then case-insensitively for the rest.

In [23]:
b = tf.groupby("bridge").agg(rows=("tx_hash", "size"), usd_provided=("amount_usd_provided", "sum")).reset_index()
b = b.merge(tools.rename(columns={"key": "bridge", "name": "tool_name"}), on="bridge", how="left")
b["match"] = b["kind"].fillna("none")
ci = tools.assign(key_lc=tools["key"].str.lower()).rename(columns={"key": "key_ci", "name": "name_ci", "kind": "kind_ci"})
b["bridge_lc"] = b["bridge"].str.lower()
b = b.merge(ci[["key_lc", "key_ci", "name_ci", "kind_ci"]], left_on="bridge_lc", right_on="key_lc", how="left").drop(columns=["key_lc", "bridge_lc"])
b["row_share"] = (b["rows"] / len(tf)).round(4)
print("bridge strings:", len(b), " matched:", b["match"].value_counts().to_dict())
print("rows by match:", b.groupby("match")["rows"].sum().to_dict(), " USD by match:", b.groupby("match")["usd_provided"].sum().round(2).to_dict())
display(b.sort_values("rows", ascending=False)[["bridge", "match", "tool_name", "n_supported_chains", "rows", "row_share", "usd_provided", "key_ci", "kind_ci"]].round(2))
print("tools in the list with no traffic in the sample:")
for kind, keys in tools[~tools["key"].isin(b["bridge"])].groupby("kind")["key"]:
    print(f"  {kind} ({len(keys)}): {', '.join(sorted(keys))}")

bridge strings: 31  matched: {'bridge': 28, 'exchange': 3}
rows by match: {'bridge': 124563, 'exchange': 295}  USD by match: {'bridge': 307709549.05, 'exchange': 226443.54}


,bridge,match,tool_name,n_supported_chains,rows,row_share,usd_provided,key_ci,kind_ci
0,across,bridge,AcrossV4,332,34163,0.27,2.471787e+07,across,bridge
26,relaydepository,bridge,Relay,1681,31483,0.25,3.402573e+07,relaydepository,bridge
10,lifiIntents,bridge,LI.FI Intents,714,16185,0.13,4.016717e+07,lifiIntents,bridge
9,layerswap,bridge,Layerswap,1849,12659,0.10,7.577968e+06,layerswap,bridge
14,mayan,bridge,Mayan (Swift),128,11422,0.09,1.919399e+07,mayan,bridge
25,polymerStandard,bridge,Polymer (Standard),439,6631,0.05,1.297023e+08,polymerStandard,bridge
6,gasZipBridge,bridge,GasZip,3132,3678,0.03,2.030286e+04,gasZipBridge,bridge
19,near,bridge,NearIntents,238,2814,0.02,1.136946e+07,near,bridge
24,polymer,bridge,Polymer (Fast),439,1320,0.01,2.017418e+06,polymer,bridge
30,symbiosis,bridge,Symbiosis,846,772,0.01,2.678868e+06,symbiosis,bridge


tools in the list with no traffic in the sample:
  bridge (10): allbridge, cctp, centrifuge, eco, garden, hyperliquidNative, hyperliquidSA, smartDeposits, stargateV2Bus, unit
  exchange (38): 1inch, aftermath, bebop, bitget, bluefin7k, cetus, dflow, dodo, eisen, enso, fly, gluex, hyperbloom, hyperflow, hyperliquidSpotDex, hypertrade, jupiter, kumbaya, kuru, kyberswap, lifidexaggregator, liquidswap, merkle, momentum, monorail, nexroute, nordstern, okx, openocean, paraswap, peach, rialto, smartDeposits, soroswap, sunswap, supersetDex, sushiswap, titan


## 10. The status sample: completion, amounts, gas, fees, tools

Everything in this section is reference data from the sampled `/v1/status` responses (§5): 100 events per source chain. Rates are shares of the sample; a 95% Wilson interval is given for the headline rate. Row-weighted means weighting each chain's rate by its share of rows in the data.

In [24]:
import math

ss = status_sample.copy()
for c in ["sending_amountUSD", "receiving_amountUSD", "sending_gasAmountUSD", "receiving_gasAmountUSD", "quote_fromAmountUSD", "quote_toAmountUSD", "quote_fee_usd", "quote_fee_lifi_usd", "quote_gas_usd", "amount_usd_provided"]:
    ss[c] = pd.to_numeric(ss[c], errors="coerce")


def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return round(c - h, 4), round(c + h, 4)


ss["done"] = ss["status"] == "DONE"
ss["completed"] = ss["done"] & (ss["substatus"] == "COMPLETED")
n, k_done, k_comp = len(ss), int(ss["done"].sum()), int(ss["completed"].sum())
print(f"HTTP statuses: {ss['http_status'].value_counts().to_dict()}")
print(f"status DONE:                          {k_done} of {n} = {k_done / n:.4f}, Wilson 95% {wilson(k_done, n)}")
print(f"status DONE and substatus COMPLETED:  {k_comp} of {n} = {k_comp / n:.4f}, Wilson 95% {wilson(k_comp, n)}")
print("status × substatus:")
display(pd.crosstab(ss["status"].fillna("(none)"), ss["substatus"].fillna("(none)"), margins=True))
by_chain = ss.groupby("chain").agg(sampled=("done", "size"), done=("done", "sum"), completed=("completed", "sum"))
by_chain["completed_rate"] = (by_chain["completed"] / by_chain["sampled"]).round(4)
by_chain["rows_in_data"] = tf["chain"].value_counts()
by_chain["row_share"] = by_chain["rows_in_data"] / len(tf)
print(f"row-weighted completed rate (each chain's rate weighted by its share of rows in the data): {(by_chain['completed'] / by_chain['sampled'] * by_chain['row_share']).sum():.4f}")
display(by_chain.round(4))
print("DONE but not COMPLETED, with the message:")
display(ss[~ss["completed"]][["chain", "block_time", "event_bridge", "event_integrator", "event_destinationChainId", "receiving_chainId", "status", "substatus", "substatusMessage", "sending_amountUSD", "receiving_amountUSD"]].round(2))

HTTP statuses: {200: 600}
status DONE:                          600 of 600 = 1.0000, Wilson 95% (0.9936, 1.0)
status DONE and substatus COMPLETED:  593 of 600 = 0.9883, Wilson 95% (0.9761, 0.9943)
status × substatus:


substatus,COMPLETED,PARTIAL,REFUNDED,All
status,,,,
DONE,593,1,6,600
All,593,1,6,600


row-weighted completed rate (each chain's rate weighted by its share of rows in the data): 0.9836


,sampled,done,completed,completed_rate,rows_in_data,row_share
chain,,,,,,
arbitrum,100,100,99,0.99,20398,0.1634
base,100,100,98,0.98,34908,0.2796
bnb,100,100,99,0.99,27052,0.2167
ethereum,100,100,97,0.97,29351,0.2351
optimism,100,100,100,1.00,5756,0.0461
polygon,100,100,100,1.00,7393,0.0592


DONE but not COMPLETED, with the message:


,chain,block_time,event_bridge,event_integrator,event_destinationChainId,receiving_chainId,status,substatus,substatusMessage,sending_amountUSD,receiving_amountUSD
41,arbitrum,2026-10-15T08:08:16+00:00,symbiosis,jumper.exchange,56,56,DONE,PARTIAL,Some of the received tokens are not the requested destin...,1941.18,1938.15
126,base,2026-10-14T14:45:05+00:00,mayan,base-app,56,8453,DONE,REFUNDED,The tokens were refunded to the user.,49.35,48.85
130,base,2026-10-14T20:20:55+00:00,relaydepository,base-app,56,8453,DONE,REFUNDED,The tokens were refunded to the user.,0.50,0.50
220,bnb,2026-10-14T15:03:37+00:00,layerswap,safepal,137,56,DONE,REFUNDED,The tokens were refunded to the user.,1.00,0.99
331,ethereum,2026-10-14T23:09:35+00:00,lifiIntents,jumper.exchange,5042,1,DONE,REFUNDED,The tokens were refunded to the user.,2526.07,2525.46
342,ethereum,2026-10-15T17:11:11+00:00,relaydepository,jumper.exchange,137,1,DONE,REFUNDED,The tokens were refunded to the user.,10.00,9.94
362,ethereum,2026-10-17T03:01:59+00:00,relaydepository,metamask-bridge,1151111081099710,1,DONE,REFUNDED,The tokens were refunded to the user.,462.74,462.17


In [25]:
by_bridge = ss.groupby("event_bridge").agg(sampled=("done", "size"), done=("done", "sum"), completed=("completed", "sum"))
by_bridge["completed_rate"] = (by_bridge["completed"] / by_bridge["sampled"]).round(4)
display(by_bridge.sort_values("sampled", ascending=False))
# Identity checks: does the response describe the sampled event?
ss["sending_hash_matches"] = ss["sending_txHash"].str.lower() == ss["tx_hash"].str.lower()
ss["transactionId_matches"] = ss["api_transactionId"].str.lower() == ss["event_transactionId"].str.lower()
ss["integrator_matches"] = ss["metadata_integrator"] == ss["event_integrator"]
ss["sending_chain_matches"] = ss["sending_chainId"].astype("Int64") == ss["chain"].map(SOURCE_CHAIN_ID).astype("Int64")
ss["dest_chain_matches"] = ss["receiving_chainId"].astype("Int64") == ss["event_destinationChainId"].astype("Int64")
print({c: int(ss[c].fillna(False).sum()) for c in ["sending_hash_matches", "transactionId_matches", "integrator_matches", "sending_chain_matches", "dest_chain_matches"]}, "of", n)
print("receiving chain id absent (no receiving leg in the response):", int(ss["receiving_chainId"].isna().sum()))
print("transactionId mismatches (two-event transactions or a different id):")
display(ss[~ss["transactionId_matches"].fillna(False)][["chain", "tx_hash", "log_index", "event_transactionId", "api_transactionId", "status", "event_bridge", "tool"]])
print("integrator mismatches:")
display(ss[~ss["integrator_matches"].fillna(False)][["chain", "event_integrator", "metadata_integrator", "quote_integrator", "status"]].value_counts().to_frame("rows"))
print("receiving chain differs from the event's destinationChainId:")
display(ss[~ss["dest_chain_matches"].fillna(False)][["chain", "event_bridge", "event_destinationChainId", "receiving_chainId", "status", "substatus", "receiving_token_symbol", "sending_amountUSD", "receiving_amountUSD"]])

,sampled,done,completed,completed_rate
event_bridge,,,,
across,179,179,179,1.0000
relaydepository,128,128,125,0.9766
lifiIntents,78,78,77,0.9872
layerswap,67,67,66,0.9851
mayan,54,54,53,0.9815
polymerStandard,36,36,36,1.0000
gasZipBridge,15,15,15,1.0000
near,15,15,15,1.0000
polymer,10,10,10,1.0000


{'sending_hash_matches': 600, 'transactionId_matches': 599, 'integrator_matches': 600, 'sending_chain_matches': 600, 'dest_chain_matches': 589} of 600
receiving chain id absent (no receiving leg in the response): 0
transactionId mismatches (two-event transactions or a different id):


,chain,tx_hash,log_index,event_transactionId,api_transactionId,status,event_bridge,tool
114,base,0x14690865381280b1a5c1b87b162e1de96e08cd8931c3175f88a293...,270,0x84f3bd4fed141d4493acfb2a76a0ff2ea7f0f5102abc62c6f620cf...,0xdc17b6a3d5001550fe126556ce0a03662b54768c009d58c12bc286...,DONE,relaydepository,relaydepository


integrator mismatches:


,,,,,rows
chain,event_integrator,metadata_integrator,quote_integrator,status,


receiving chain differs from the event's destinationChainId:


,chain,event_bridge,event_destinationChainId,receiving_chainId,status,substatus,receiving_token_symbol,sending_amountUSD,receiving_amountUSD
126,base,mayan,56,8453,DONE,REFUNDED,USDC,49.3491,48.8482
130,base,relaydepository,56,8453,DONE,REFUNDED,cbBTC,0.4991,0.4961
195,base,near,1885080386571452,728126428,DONE,COMPLETED,USDT,111.3827,103.1457
220,bnb,layerswap,137,56,DONE,REFUNDED,USDT,0.9995,0.9927
306,ethereum,near,1885080386571452,728126428,DONE,COMPLETED,TRX,199.4567,199.6194
324,ethereum,near,1885080386571452,728126428,DONE,COMPLETED,USDT,17798.4094,17766.7407
331,ethereum,lifiIntents,5042,1,DONE,REFUNDED,WBTC,2526.0724,2525.4557
342,ethereum,relaydepository,137,1,DONE,REFUNDED,USDC,9.9986,9.9406
362,ethereum,relaydepository,1151111081099710,1,DONE,REFUNDED,USDC,462.7437,462.1683
391,ethereum,near,1885080386571452,728126428,DONE,COMPLETED,USDT,55.4217,51.8036


In [26]:
# The tool the API reports against the bridge string in the event.
ss["tool_matches_bridge"] = ss["tool"] == ss["event_bridge"]
print("tool == event bridge:", int(ss["tool_matches_bridge"].sum()), "of", n, "; tool missing:", int(ss["tool"].isna().sum()))
display(ss[~ss["tool_matches_bridge"]].groupby(["event_bridge", ss["tool"].fillna("(none)")]).size().to_frame("rows"))
print("included source-side steps (swaps and fee collection) by whether the event flags a source swap:")
ss["has_included_tools"] = ss["sending_included_tools"].fillna("") != ""
display(pd.crosstab(ss["event_hasSourceSwaps"], ss["has_included_tools"]))
display(ss["sending_included_tools"].fillna("").str.split("|").explode().replace("", "(none)").value_counts().head(12).to_frame("steps"))

tool == event bridge: 600 of 600 ; tool missing: 0


,,rows
event_bridge,tool,


included source-side steps (swaps and fee collection) by whether the event flags a source swap:


has_included_tools,False,True
event_hasSourceSwaps,,
False,198,0
True,0,402


,steps
sending_included_tools,
feeCollection,355
(none),198
sushiswap,41
nordstern,31
okx,27
bitget,23
wrapper,11
kyberswap,11
1inch,7


In [27]:
# Receiving versus sending USD on completed transfers, and the API's sending USD against the provided-feed column for the same events.
done_ = ss[ss["done"] & (ss["substatus"] == "COMPLETED")].copy()
done_["recv_over_send"] = done_["receiving_amountUSD"] / done_["sending_amountUSD"]
done_["send_over_provided"] = done_["sending_amountUSD"] / done_["amount_usd_provided"]
print("completed transfers with both USD amounts:", int(done_["recv_over_send"].notna().sum()), "of", len(done_))
display(done_["recv_over_send"].describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]).to_frame("receiving_usd / sending_usd").T.round(4))
loss = 1 - done_["recv_over_send"]
print(f"median USD lost between sending and receiving: {loss.median():.4%}; share losing more than 1%: {(loss > 0.01).mean():.4f}; more than 5%: {(loss > 0.05).mean():.4f}; receiving more than sent: {(loss < 0).mean():.4f}")
print(f"sum sending USD {done_['sending_amountUSD'].sum():,.2f}  sum receiving USD {done_['receiving_amountUSD'].sum():,.2f}  ratio {done_['receiving_amountUSD'].sum() / done_['sending_amountUSD'].sum():.4f}")
display(done_.groupby("event_bridge")["recv_over_send"].agg(n="size", median="median", p10=lambda s: s.quantile(0.1), min="min").query("n >= 10").round(4).sort_values("n", ascending=False))
print("completed transfers whose receiving USD is below half or above twice the sending USD (the API's USD on the receiving token, not a loss in the transfer):")
display(done_[(done_["recv_over_send"] < 0.5) | (done_["recv_over_send"] > 2)][["chain", "event_bridge", "event_destinationChainId", "sending_token_symbol", "sending_amountUSD", "receiving_token_symbol", "receiving_amount", "receiving_token_decimals", "receiving_amountUSD", "recv_over_send"]].round(3))
print("the same ratio without those rows:")
trimmed = done_[(done_["recv_over_send"] >= 0.5) & (done_["recv_over_send"] <= 2)]["recv_over_send"]
print(f"n {len(trimmed)}, median {trimmed.median():.4f}, mean {trimmed.mean():.4f}, p5 {trimmed.quantile(0.05):.4f}, p95 {trimmed.quantile(0.95):.4f}")
print("the API's sending USD against the provided-feed column (ratio), split by the source-swap flag: with a swap the API's sending amount is the pre-swap input, the event's is the bridged token")
display(done_.groupby("event_hasSourceSwaps")["send_over_provided"].describe(percentiles=[0.05, 0.5, 0.95]).round(4))
print("largest disagreements between the API's sending USD and the provided column:")
display(done_.assign(abs_dev=(done_["send_over_provided"] - 1).abs()).sort_values("abs_dev", ascending=False).head(8)[["chain", "event_bridge", "event_hasSourceSwaps", "sending_token_symbol", "sending_amountUSD", "amount_usd_provided", "send_over_provided"]].round(3))

completed transfers with both USD amounts: 593 of 593


,count,mean,std,min,5%,10%,25%,50%,75%,90%,95%,max
receiving_usd / sending_usd,593.0,1.0473,1.2925,0.0,0.9242,0.9654,0.9896,0.997,0.9997,1.0017,1.0032,31.3287


median USD lost between sending and receiving: 0.3048%; share losing more than 1%: 0.2580; more than 5%: 0.0776; receiving more than sent: 0.2125
sum sending USD 2,720,583.38  sum receiving USD 2,718,743.50  ratio 0.9993


,n,median,p10,min
event_bridge,,,,
across,179,0.9975,0.9850,0.9449
relaydepository,125,0.9872,0.8120,0.0000
lifiIntents,77,0.9990,0.9931,0.9142
layerswap,66,0.9962,0.9740,0.5476
mayan,53,0.9903,0.9654,0.9367
polymerStandard,36,1.0003,0.9974,0.9937
gasZipBridge,15,0.9960,0.9878,0.9844
near,15,0.9977,0.9539,0.9260
polymer,10,0.9971,0.9747,0.9360


completed transfers whose receiving USD is below half or above twice the sending USD (the API's USD on the receiving token, not a loss in the transfer):


,chain,event_bridge,event_destinationChainId,sending_token_symbol,sending_amountUSD,receiving_token_symbol,receiving_amount,receiving_token_decimals,receiving_amountUSD,recv_over_send
35,arbitrum,relaydepository,1151111081099710,USDC,6.929,WAP,50673704404,6,1.292,0.186
137,base,across,4663,USDC,10.052,GMOTH,9115839140625927251094305,18,91.561,9.109
207,bnb,relaydepository,4663,USDT,5.598,OMF,794131495425327265812879,18,0.000,0.000
222,bnb,relaydepository,4663,USDT,49.975,BBC,48049309760,6,0.000,0.000
307,ethereum,across,4663,USDC,2.595,RMOLE,3424013117325037494301261,18,81.301,31.329


the same ratio without those rows:
n 588, median 0.9970, mean 0.9871, p5 0.9294, p95 1.0031
the API's sending USD against the provided-feed column (ratio), split by the source-swap flag: with a swap the API's sending amount is the pre-swap input, the event's is the bridged token


,count,mean,std,min,5%,50%,95%,max
event_hasSourceSwaps,,,,,,,,
False,198.0,0.9996,0.0021,0.9877,0.9962,0.9998,1.0027,1.0052
True,392.0,1.0055,0.0128,0.9847,0.9978,1.0026,1.0205,1.1334


largest disagreements between the API's sending USD and the provided column:


,chain,event_bridge,event_hasSourceSwaps,sending_token_symbol,sending_amountUSD,amount_usd_provided,send_over_provided
293,bnb,relaydepository,True,USDC,7.818,6.898,1.133
54,arbitrum,relaydepository,True,unshETH,2.072,1.858,1.115
379,ethereum,mayanMCTP,True,ANYONE,16.473,15.369,1.072
195,base,near,True,LUNA,111.383,104.548,1.065
273,bnb,mayan,True,ALEO,104.240,97.916,1.065
561,polygon,mayan,True,LDO,451.278,425.591,1.060
169,base,mayan,True,BASEPAD,5.688,5.395,1.054
391,ethereum,near,True,PUFFER,55.422,52.826,1.049


In [28]:
# Gas USD on both legs and the quote's fee costs (what the fee model in the brief calls LI.FI's and the integrator's fees).
print("gas USD, sending leg:")
display(ss["sending_gasAmountUSD"].describe(percentiles=[0.5, 0.9, 0.99]).to_frame().T.round(4))
display(ss.groupby("chain")["sending_gasAmountUSD"].agg(n="count", median="median", mean="mean", total="sum").round(4))
print("gas USD, receiving leg (where present):")
display(ss["receiving_gasAmountUSD"].describe(percentiles=[0.5, 0.9, 0.99]).to_frame().T.round(4))
print("quote fee costs: rows with a quote and with fee costs:", int(ss["quote_tool"].notna().sum()), int(ss["quote_fee_usd"].notna().sum()))
display(ss[["quote_fee_usd", "quote_fee_lifi_usd", "quote_gas_usd"]].describe(percentiles=[0.5, 0.9, 0.99]).round(4))
ss["fee_share_of_sending"] = ss["quote_fee_usd"] / ss["sending_amountUSD"]
ss["lifi_fee_share_of_sending"] = ss["quote_fee_lifi_usd"] / ss["sending_amountUSD"]
print(f"fee USD over sending USD, median {ss['fee_share_of_sending'].median():.4%}, LI.FI-named fee median {ss['lifi_fee_share_of_sending'].median():.4%}; "
      f"rows with a LI.FI-named fee > 0: {int((ss['quote_fee_lifi_usd'] > 0).sum())}; sum of all fee USD in the sample {ss['quote_fee_usd'].sum():,.2f}, LI.FI-named {ss['quote_fee_lifi_usd'].sum():,.2f}")
display(ss["quote_fee_names"].fillna("").str.split("|").explode().replace("", "(none)").value_counts().head(15).to_frame("fee_lines"))
display(ss.groupby("event_bridge").agg(n=("quote_fee_usd", "size"), fee_usd=("quote_fee_usd", "sum"), lifi_fee_usd=("quote_fee_lifi_usd", "sum"), sending_usd=("sending_amountUSD", "sum")).assign(fee_bps=lambda d: (1e4 * d["fee_usd"] / d["sending_usd"]).round(1), lifi_fee_bps=lambda d: (1e4 * d["lifi_fee_usd"] / d["sending_usd"]).round(1)).sort_values("n", ascending=False).round(2))

gas USD, sending leg:


,count,mean,std,min,50%,90%,99%,max
sending_gasAmountUSD,600.0,0.0609,0.2657,0.0001,0.01,0.1,0.9099,3.3551


,n,median,mean,total
chain,,,,
arbitrum,100,0.0106,0.0167,1.6749
base,100,0.0032,0.0062,0.6222
bnb,100,0.0154,0.0258,2.5768
ethereum,100,0.1176,0.2982,29.8220
optimism,100,0.0001,0.0004,0.0441
polygon,100,0.0106,0.0182,1.8156


gas USD, receiving leg (where present):


,count,mean,std,min,50%,90%,99%,max
receiving_gasAmountUSD,600.0,0.0272,0.113,0.0,0.0047,0.045,0.3115,2.2485


quote fee costs: rows with a quote and with fee costs: 599 511


,quote_fee_usd,quote_fee_lifi_usd,quote_gas_usd
count,511.0000,511.0000,599.0000
mean,0.7784,0.5442,0.0276
std,3.9347,3.2514,0.0709
min,0.0001,0.0000,0.0001
50%,0.0564,0.0054,0.0085
90%,1.5454,0.9981,0.0578
99%,7.3032,7.0083,0.3857
max,50.3727,49.6321,0.8670


fee USD over sending USD, median 0.2765%, LI.FI-named fee median 0.0500%; rows with a LI.FI-named fee > 0: 353; sum of all fee USD in the sample 397.77, LI.FI-named 278.09


,fee_lines
quote_fee_names,
LIFI Fixed Fee,287
Relayer fee,179
Relayer gas fee,179
Relayer Service fee,127
Relayer Gas fee,124
(none),89
LayerSwap fee,67
LIFI Shared Fee,66
LP fee,40


,n,fee_usd,lifi_fee_usd,sending_usd,fee_bps,lifi_fee_bps
event_bridge,,,,,,
across,179,116.30,101.49,96777.60,12.0,10.5
relaydepository,128,90.01,15.34,527011.25,1.7,0.3
lifiIntents,78,72.28,71.94,116870.29,6.2,6.2
layerswap,67,31.96,20.99,8818.59,36.2,23.8
mayan,54,57.01,57.01,564000.40,1.0,1.0
polymerStandard,36,0.90,0.90,176694.23,0.1,0.1
gasZipBridge,15,0.50,0.50,107.10,47.0,47.0
near,15,15.82,6.62,20230.44,7.8,3.3
polymer,10,1.35,0.41,644.27,20.9,6.4


## 11. Manifest

`data/reference/snapshot_manifest.json` as written by the fetch cells above; `data/reference/README.md` restates it with one row per file.

In [29]:
m = manifest_read()
display(pd.DataFrame.from_dict(m, orient="index")[["fetched_at_utc", "rows", "notebook_section", "source_url"]].sort_index())
for name, entry in sorted(m.items()):
    if entry.get("note"):
        print(f"{name}: {entry['note']}")
print("files under data/reference:", sorted(p.name for p in REF.iterdir()))

,fetched_at_utc,rows,notebook_section,source_url
coingecko_hourly_prices_2026-10-13_to_2026-10-19.csv,2026-10-28T16:06:54+00:00,41777,01 §6,https://api.coingecko.com/api/v3/coins/<platform>/contra...
coingecko_request_log.csv,2026-10-28T16:06:54+00:00,256,01 §6,https://api.coingecko.com/api/v3/coins/<platform>/contra...
lifi_api_chains.csv,2026-10-28T14:48:32+00:00,70,01 §2,"https://li.quest/v1/chains?chainTypes=EVM,SVM,UTXO,MVM,TVM"
lifi_api_chains_with_stl_2026-10-28.csv,2026-10-28T21:35:06+00:00,71,01 §2.1,"https://li.quest/v1/chains?chainTypes=EVM,SVM,UTXO,MVM,T..."
lifi_api_tokens_by_chain.csv,2026-10-28T14:48:32+00:00,35903,01 §3,"https://li.quest/v1/tokens?chains=<id>&chainTypes=EVM,SV..."
lifi_api_tools_bridges_and_exchanges.csv,2026-10-28T14:48:56+00:00,79,01 §4,https://li.quest/v1/tools
lifi_api_transfer_status_edge_cases.csv,2026-10-28T19:01:00+00:00,328,03 §4,https://li.quest/v1/status?txHash=<tx_hash> (lookup tx_h...
lifi_api_transfer_status_edge_cases_raw.jsonl,2026-10-28T19:01:00+00:00,311,03 §4,https://li.quest/v1/status?txHash=<tx_hash> (lookup tx_h...
lifi_api_transfer_status_sample_2600.csv,2026-10-28T19:07:08+00:00,2600,03 §6,https://li.quest/v1/status?txHash=<tx_hash>
lifi_api_transfer_status_sample_2600_raw.jsonl,2026-10-28T19:07:08+00:00,2600,03 §6,https://li.quest/v1/status?txHash=<tx_hash>


coingecko_hourly_prices_2026-10-13_to_2026-10-19.csv: 256 requests for 255 (chain, token) pairs; pairs with at least one hour: 249; HTTP statuses {200: 250, 404: 6}; key used: True; request log in coingecko_request_log.csv
coingecko_request_log.csv: one row per CoinGecko request: chain, token, request URL (no key), HTTP status, points returned, hours inside the window
lifi_api_chains.csv: chainTypes names every value the endpoint accepts; without it only EVM chains are returned. Correction 2026-10-28: the parameter also accepts STL, which this fetch did not pass, so Stellar is missing; see lifi_api_chains_with_stl_2026-10-28.csv
lifi_api_chains_with_stl_2026-10-28.csv: chainTypes with all six values LI.FI's API reference lists (EVM, SVM, UTXO, MVM, TVM, STL); lifi_api_chains.csv (§2) passed five and lacks Stellar; derived from lifi_chain_id_registry_sources_raw.jsonl at 2026-10-28T21:47:47+00:00
lifi_api_tokens_by_chain.csv: one /v1/tokens call per chain id for 69 ids (6 source + desti